# SWAC Football Recruitment Data Processing Pipeline

**Production-Ready Analysis of Player State Assignments**

This notebook processes SWAC football recruitment data to achieve 99.4%+ completion rate for player state assignments through systematic data cleaning and corrections.

## Overview
- **Dataset**: SWAC_Rosters_Combined.csv (15,571 players, 2010-2022)
- **Goal**: Extract player home state from messy hometown data
- **Target**: 99.36%+ completion rate
- **Method**: Automated extraction + systematic manual corrections

## Key Features
- Comprehensive state extraction from AP-style, USPS, and full state names
- International player detection and classification
- Pattern-based corrections for common data inconsistencies
- Individual player corrections for maximum accuracy
- Complete audit trail and success rate tracking

In [16]:
# ===== SECTION 1: DATA LOADING AND INITIAL SETUP =====

import pandas as pd
import numpy as np
import re

# Load the dataset
print("Loading SWAC Football Recruitment Dataset...")
swac_fb = pd.read_csv("SWAC_Rosters_Combined.csv")
swac_fb = swac_fb[['team', 'season', 'name', 'high_school', 'hometown', 'previous_school', 'class']]

print(f"✅ Dataset loaded successfully!")
print(f"   📊 Total records: {len(swac_fb):,}")
print(f"   🏫 Teams: {', '.join(sorted(swac_fb['team'].unique()))}")
print(f"   📅 Seasons: {swac_fb['season'].min()}-{swac_fb['season'].max()}")
print(f"   📋 Columns: {list(swac_fb.columns)}")

# Initial data quality assessment
print(f"\n📈 Initial Data Quality:")
print(f"   🏠 Hometown data: {swac_fb['hometown'].notna().sum():,} non-null ({swac_fb['hometown'].notna().sum()/len(swac_fb)*100:.1f}%)")
print(f"   🎓 High school data: {swac_fb['high_school'].notna().sum():,} non-null ({swac_fb['high_school'].notna().sum()/len(swac_fb)*100:.1f}%)")
print(f"   📚 Previous school data: {swac_fb['previous_school'].notna().sum():,} non-null ({swac_fb['previous_school'].notna().sum()/len(swac_fb)*100:.1f}%)")

swac_fb.head()

Loading SWAC Football Recruitment Dataset...
✅ Dataset loaded successfully!
   📊 Total records: 15,571
   🏫 Teams: Alabama A&M, Alabama State, Alcorn State, Bethune-Cookman, Florida A&M, Grambling, Jackson State, Mississippi Valley State, Prairie View A&M, Southern, Texas Southern, UAPB
   📅 Seasons: 2010-2025
   📋 Columns: ['team', 'season', 'name', 'high_school', 'hometown', 'previous_school', 'class']

📈 Initial Data Quality:
   🏠 Hometown data: 15,545 non-null (99.8%)
   🎓 High school data: 7,205 non-null (46.3%)
   📚 Previous school data: 7,849 non-null (50.4%)


,team,season,name,high_school,hometown,previous_school,class
0,Jackson State,2025,Travis Terrell Jr.,Creekside HS,"Atlanta, Ga.","Atlanta, Ga. /",So.
1,Jackson State,2025,Jeremiah Williams,Holmes County Central HS,"Lexington, Miss.","Lexington, Miss. /",R-Sr.
2,Jackson State,2025,Khamauri Rogers,Holmes County Central HS,"Madison, Miss.","Madison, Miss. / Mississippi State",Gr.
3,Jackson State,2025,Shemar Savage,Lompoc HS,"Lompoc, Calif.","Lompoc, Calif. / Prairie View A&M",Gr.
4,Jackson State,2025,Ja'Naylon Dupree,Neshoba Central HS,"Philadelphia, Miss.","Philadelphia, Miss. / Mississippi Gulf Coast CC",Sr.


In [17]:
# ===== SECTION 2: DATA CLEANING FUNCTIONS =====

def clean_hometown(hometown):
    """Remove trailing slashes and extra whitespace from hometown data."""
    if pd.isna(hometown) or not isinstance(hometown, str):
        return hometown
    
    # Remove trailing slashes and strip whitespace
    cleaned = hometown.rstrip('/').strip()
    
    # Remove any double spaces that might result
    cleaned = ' '.join(cleaned.split())
    
    return cleaned if cleaned else None

def clean_previous_school(x):
    """Clean previous school column to remove repeated hometown data."""
    if not isinstance(x, str) or not x.strip():
        return None
    
    # Case 1: if there's a slash, keep only the part after it
    if '/' in x:
        x = x.split('/')[-1].strip()
    
    # Case 2: if the remaining text looks like a city/state (e.g., "Atlanta, Ga.")
    # Remove it by returning None
    if re.match(r'^[A-Za-z\s\.-]+,\s*[A-Za-z\.]{2,}$', x.strip()):
        return None
    
    return x.strip()

# Apply cleaning functions
print("🧹 Applying data cleaning functions...")
swac_fb['hometown'] = swac_fb['hometown'].apply(clean_hometown)
swac_fb['previous_school'] = swac_fb['previous_school'].apply(clean_previous_school)

print("✅ Data cleaning completed!")
print(f"   🏠 Cleaned hometown entries: {swac_fb['hometown'].notna().sum():,}")
print(f"   📚 Cleaned previous school entries: {swac_fb['previous_school'].notna().sum():,}")

# Display sample of cleaned data
print(f"\n📋 Sample of cleaned data:")
swac_fb[['name', 'team', 'hometown', 'high_school', 'previous_school']].head(10)

🧹 Applying data cleaning functions...
✅ Data cleaning completed!
   🏠 Cleaned hometown entries: 15,497
   📚 Cleaned previous school entries: 7,839

📋 Sample of cleaned data:


,name,team,hometown,high_school,previous_school
0,Travis Terrell Jr.,Jackson State,"Atlanta, Ga.",Creekside HS,
1,Jeremiah Williams,Jackson State,"Lexington, Miss.",Holmes County Central HS,
2,Khamauri Rogers,Jackson State,"Madison, Miss.",Holmes County Central HS,Mississippi State
3,Shemar Savage,Jackson State,"Lompoc, Calif.",Lompoc HS,Prairie View A&M
4,Ja'Naylon Dupree,Jackson State,"Philadelphia, Miss.",Neshoba Central HS,Mississippi Gulf Coast CC
5,Levi Wyatt,Jackson State,"Vicksburg, Miss.",Holmes County Central HS,McNeese State
6,Nate Rembert,Jackson State,"Eustis, Fla.",Eustis HS,Mississippi Valley State
7,Ashton Taylor,Jackson State,"Hoover, Ala.",Hoover HS,Tennessee Tech
8,Tyquan Henderson,Jackson State,"Canton, Miss.",Canton HS,Southern Miss
9,Mike Smith III,Jackson State,"Dayton, Ohio",Trotwood-Mason HS,


In [18]:
# ===== SECTION 3: STATE MAPPING DICTIONARIES AND EXTRACTION LOGIC =====

# Comprehensive mapping for AP-style abbreviations to USPS codes
ap_to_usps = {
    # Standard AP-style state abbreviations
    'Ala.':'AL', 'Ala':'AL', 'ALA.':'AL', 'ALA':'AL',
    'Ariz.':'AZ', 'Ariz':'AZ', 'ARIZ.':'AZ', 'ARIZ':'AZ',
    'Ark.':'AR', 'Ark':'AR', 'ARK.':'AR', 'ARK':'AR',
    'Cal.':'CA', 'Cal':'CA', 'CAL.':'CA', 'CAL':'CA',
    'Calif.':'CA', 'Calif':'CA', 'CALIF.':'CA', 'CALIF':'CA',
    'Ca.': 'CA', 'Ca': 'CA', 'CA.':'CA', 'CA':'CA',
    'Colo.':'CO', 'Colo':'CO', 'COLO.':'CO', 'COLO':'CO',
    'Conn.':'CT', 'Conn':'CT', 'CONN.':'CT', 'CONN':'CT',
    'Del.':'DE', 'Del':'DE', 'DEL.':'DE', 'DEL':'DE',
    'Fla.':'FL', 'Fla':'FL', 'FLA.':'FL', 'FLA':'FL',
    'Ga.':'GA', 'Ga': 'GA', 'GA.':'GA', 'GA':'GA',
    'Ill.':'IL', 'Ill':'IL', 'ILL.':'IL', 'ILL':'IL',
    'Ind.':'IN', 'Ind':'IN', 'IND.':'IN', 'IND':'IN',
    'Kan.':'KS', 'Kan':'KS', 'KAN.':'KS', 'KAN':'KS', 'Kans.':'KS', 'Kans':'KS',
    'Ky.':'KY', 'Ky':'KY', 'KY.':'KY', 'KY':'KY',
    'La.':'LA', 'La': 'LA', 'LA.':'LA', 'LA':'LA',
    'Md.':'MD', 'Md':'MD', 'MD.':'MD', 'MD':'MD',
    'Mass.':'MA', 'Mass':'MA', 'MASS.':'MA', 'MASS':'MA',
    'Mich.':'MI', 'Mich':'MI', 'MICH.':'MI', 'MICH':'MI',
    'Minn.':'MN', 'Minn':'MN', 'MINN.':'MN', 'MINN':'MN',
    'Miss.':'MS', 'Miss':'MS', 'MISS.':'MS', 'MISS':'MS',
    'Mo.':'MO', 'Mo':'MO', 'MO.':'MO', 'MO':'MO',
    'Mont.':'MT', 'Mont':'MT', 'MONT.':'MT', 'MONT':'MT',
    'Neb.':'NE', 'Neb':'NE', 'NEB.':'NE', 'NEB':'NE', 'Nebr.':'NE', 'Nebr':'NE',
    'Nev.':'NV', 'Nev':'NV', 'NEV.':'NV', 'NEV':'NV',
    'N.H.':'NH', 'NH.':'NH', 'NH':'NH',
    'N.J.':'NJ', 'NJ.':'NJ', 'NJ':'NJ',
    'N.M.':'NM', 'NM.':'NM', 'NM':'NM',
    'N.Y.':'NY', 'NY.':'NY', 'NY':'NY',
    'N.C.':'NC', 'NC.':'NC', 'NC':'NC',
    'N.D.':'ND', 'ND.':'ND', 'ND':'ND',
    'Ohio':'OH', 'OHIO':'OH', 'Ohio.':'OH', 'OHIO.':'OH',
    'Okla.':'OK', 'Okla':'OK', 'OKLA.':'OK', 'OKLA':'OK',
    'Ore.':'OR', 'Ore':'OR', 'ORE.':'OR', 'ORE':'OR', 'Oreg.':'OR', 'Oreg':'OR',
    'Pa.':'PA', 'Pa':'PA', 'PA.':'PA', 'PA':'PA',
    'Penn.':'PA', 'Penn':'PA', 'PENN.':'PA', 'PENN':'PA',
    'Penna.':'PA', 'Penna':'PA', 'PENNA.':'PA', 'PENNA':'PA',
    'R.I.':'RI', 'RI.':'RI', 'RI':'RI',
    'S.C.':'SC', 'SC.':'SC', 'SC':'SC',
    'S.D.':'SD', 'SD.':'SD', 'SD':'SD',
    'Tenn.':'TN', 'Tenn':'TN', 'TENN.':'TN', 'TENN':'TN',
    'Texas':'TX', 'TEXAS':'TX', 'Texas.':'TX', 'TEXAS.':'TX',
    'Tx.':'TX', 'Tx':'TX', 'TX.':'TX', 'TX':'TX',
    'Tex.': 'TX', 'Tex':'TX', 'TEX.':'TX', 'TEX':'TX',
    'Utah':'UT', 'UTAH':'UT', 'Utah.':'UT', 'UTAH.':'UT',
    'Vt.':'VT', 'Vt':'VT', 'VT.':'VT', 'VT':'VT',
    'Va.':'VA', 'Va':'VA', 'VA.':'VA', 'VA':'VA',
    'Wash.':'WA', 'Wash':'WA', 'WASH.':'WA', 'WASH':'WA',
    'W.Va.':'WV', 'W.V.':'WV', 'WV.':'WV', 'WV':'WV',
    'W Va.':'WV', 'W V.':'WV', 'W.Va':'WV', 'W.V':'WV',
    'Wis.':'WI', 'Wis':'WI', 'WIS.':'WI', 'WIS':'WI',
    'Wisc.':'WI', 'Wisc':'WI', 'WISC.':'WI', 'WISC':'WI',
    'Wyo.':'WY', 'Wyo':'WY', 'WYO.':'WY', 'WYO':'WY',
    
    # Additional variations found in data
    'FL.': 'FL', 'Fl.': 'FL', 'fl.': 'FL', 'Al.': 'AL', 'al.': 'AL',
    'TN.': 'TN', 'OH.': 'OH', 'OK.': 'OK', 'WI.': 'WI', 'IL.': 'IL',
    'IN.': 'IN', 'KY.': 'KY', 'NC.': 'NC', 'SC.': 'SC', 'VA.': 'VA',
    'MD.': 'MD', 'NJ.': 'NJ', 'CT.': 'CT', 'MA.': 'MA', 'PA.': 'PA',
    'NY.': 'NY', 'CA.': 'CA', 'TX.': 'TX', 'CO.': 'CO', 'AZ.': 'AZ',
    
    # Common typos and variations
    'Fl': 'FL', 'FLa.': 'FL', 'Ga .': 'GA', 'S.C': 'SC',
    'Claif.': 'CA', 'M.d.': 'MD', 'Ari.': 'AZ', 'Tn.': 'TN',
    'Oh.': 'OH', 'Ms.': 'MS', 'LS': 'LA', 'Mi.': 'MI', 'Wi.': 'WI',
    'Az.': 'AZ'
}

# Full state names to USPS codes
name_to_usps = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO',
    'Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID',
    'Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA',
    'Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC',
    'North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA',
    'Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD','Tennessee':'TN','Texas':'TX',
    'Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV',
    'Wisconsin':'WI','Wyoming':'WY'
}

# Valid USPS state codes and team states
usps_codes = set(name_to_usps.values())
school_states = {
    'Alabama A&M':'AL', 'Alabama State':'AL', 'UAPB':'AR', 'Grambling':'LA',
    'Jackson State':'MS', 'Mississippi Valley State':'MS', 'Prairie View A&M':'TX',
    'Southern':'LA', 'Bethune-Cookman':'FL', 'Florida A&M':'FL',
    'Texas Southern':'TX', 'Alcorn State':'MS'
}

print(f"🗺️ State mapping dictionaries created:")
print(f"   📝 AP-style variations: {len(ap_to_usps)}")
print(f"   🏛️ Full state names: {len(name_to_usps)}")
print(f"   🏫 SWAC team states: {len(school_states)}")

# Create team_state column
swac_fb['team_state'] = swac_fb['team'].map(school_states)
print(f"✅ Team state mapping completed!")

🗺️ State mapping dictionaries created:
   📝 AP-style variations: 238
   🏛️ Full state names: 50
   🏫 SWAC team states: 12
✅ Team state mapping completed!


In [19]:
# ===== SECTION 4: AUTOMATED STATE EXTRACTION PIPELINE =====

def extract_state_code(s):
    """
    Extract state code from hometown string.
    Handles AP-style (Miss.), USPS (MS), and full names (Mississippi).
    Returns standardized two-letter postal abbreviation or None.
    """
    if not isinstance(s, str) or not s.strip():
        return None
    s = s.strip()

    # 1) Check for USPS 2-letter code at end
    m = re.search(r'\b([A-Z]{2})\b$', s.strip())
    if m:
        ab = m.group(1).upper()
        if ab in usps_codes:
            return ab

    # 2) Extract last word (including periods for AP-style) and map
    m = re.search(r'([A-Za-z\.]+)$', s)
    if m:
        token = m.group(1)
        if token in ap_to_usps:
            return ap_to_usps[token]
        # 3) Try full state name (remove any trailing period for this check)
        token_no_period = token.rstrip('.')
        if token_no_period in name_to_usps:
            return name_to_usps[token_no_period]

    # 4) Try multi-word full state (e.g., "New Mexico", "West Virginia")
    for name, ab in name_to_usps.items():
        if name.lower() in s.lower():
            return ab

    return None

def is_international_player(hometown):
    """Check if a hometown contains any known international countries/territories."""
    if pd.isna(hometown) or hometown == '':
        return False
    
    international_countries = [
        'Canada', 'American Samoa', 'Mexico', 'Manitoba', 
        'Australia', 'Germany', 'Cameroon', 'Brasil', 
        'Amsterdam', 'Nigeria', 'Quebec', 'Jamaica',
        'Venezuela', 'Bahamas', 'Hungary', 'Ontario',
        'Netherlands', 'Spain', 'France', 'Liberia',
        'Tasmania', 'Virgin Islands', 'V.I.'
    ]
    
    hometown_lower = str(hometown).lower()
    for country in international_countries:
        if country.lower() in hometown_lower:
            return True
    return False

def extract_state_from_parentheses_format(hometown):
    """
    Extract state from Grambling's format like "New Orleans, LA (Kennedy High School)"
    Returns the state part before the parentheses.
    """
    if not isinstance(hometown, str) or not hometown.strip():
        return None
    
    # Check for pattern: text (something in parentheses)
    match = re.match(r'^(.+?)\s*\([^)]+\)$', hometown.strip())
    if match:
        # Extract the part before parentheses and treat it as "City, State"
        city_state_part = match.group(1).strip()
        # Now apply our existing state extraction to this cleaned part
        return extract_state_code(city_state_part)
    
    return None

# Apply automated state extraction
print("🤖 Applying automated state extraction pipeline...")

# Initial state extraction
swac_fb['player_state'] = swac_fb['hometown'].apply(extract_state_code)

# Apply international detection
swac_fb['is_international'] = swac_fb['hometown'].apply(is_international_player)
swac_fb.loc[swac_fb['is_international'], 'player_state'] = 'INTL'

# Handle Grambling's parentheses format (2010-2011 seasons)
parentheses_mask = (
    swac_fb['hometown'].str.contains(r'\([^)]+\)$', na=False) & 
    (swac_fb['team'] == 'Grambling') & 
    (swac_fb['season'].isin([2010, 2011]))
)

# Extract state from parentheses format and update missing entries
missing_mask = swac_fb['player_state'].isnull()
parentheses_states = swac_fb.loc[parentheses_mask & missing_mask, 'hometown'].apply(extract_state_from_parentheses_format)
swac_fb.loc[parentheses_mask & missing_mask, 'player_state'] = parentheses_states

# Calculate initial automated results
total_players = len(swac_fb)
missing_count = swac_fb['player_state'].isnull().sum()
intl_count = (swac_fb['player_state'] == 'INTL').sum()
success_rate = ((total_players - missing_count) / total_players * 100)

print(f"✅ Automated extraction completed:")
print(f"   📊 Total players: {total_players:,}")
print(f"   ✅ Successfully assigned: {total_players - missing_count:,}")
print(f"   🌍 International players: {intl_count}")
print(f"   ❌ Missing: {missing_count}")
print(f"   📈 Success rate: {success_rate:.2f}%")

# Store baseline metrics
baseline_missing = missing_count
baseline_success_rate = success_rate

🤖 Applying automated state extraction pipeline...
✅ Automated extraction completed:
   📊 Total players: 15,571
   ✅ Successfully assigned: 15,246
   🌍 International players: 87
   ❌ Missing: 325
   📈 Success rate: 97.91%


In [20]:
# ===== SECTION 5: PATTERN-BASED CORRECTIONS =====

print("🔧 Applying systematic pattern-based corrections...")

corrections_applied = 0

# City-to-state mappings for major cities
city_state_fixes = {
    'Honolulu': 'HI', 'Tallahassee': 'FL', 'Atlanta': 'GA', 'Seattle': 'WA',
    'Benton Harbor': 'MI', 'The Woodlands': 'TX', 'Brookhaven': 'MS',
    'Grand Rapids': 'MI', 'Warner Robins': 'GA', 'Vicksburg': 'MS',
    'Fairbanks': 'AK', 'Cibolo': 'TX', 'Deerfield Beach': 'FL',
    'Duncanville': 'TX', 'Killeen': 'TX', 'Port Arthur': 'TX',
    'Port St. Lucie': 'FL', 'Sarasota': 'FL', 'Waterloo': 'IA',
    'Cincinnati': 'OH', 'Perris': 'CA', 'Detroit': 'MI',
    'New Orleans': 'LA', 'Baton Rouge': 'LA'
}

# Apply city mappings
for city, state in city_state_fixes.items():
    mask = swac_fb['hometown'].str.contains(city, na=False, case=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_applied += mask.sum()

print(f"   🏙️ City mappings: {len([k for k,v in city_state_fixes.items()])} patterns applied")

# State abbreviation and suffix corrections
suffix_corrections = {
    'Wi.': 'WI', 'Az.': 'AZ', 'Mi.': 'MI', 'Mi': 'MI',
    'Fla,.': 'FL', 'Fla,': 'FL', 'Lo.': 'LA', 'Ga,': 'GA',
    'Ak.': 'AK', 'Geo.': 'GA', 'Aus.': 'INTL', 'AU': 'INTL',
    'SK': 'INTL', 'A.S.': 'INTL', 'Ont.': 'INTL', 'Ont': 'INTL'
}

# Apply suffix corrections
for suffix, state in suffix_corrections.items():
    mask = swac_fb['hometown'].str.endswith(suffix, na=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_applied += mask.sum()

print(f"   📝 Suffix corrections: {len(suffix_corrections)} patterns applied")

# Misspelling corrections
misspelling_patterns = {
    'Forida': 'FL', 'Mississisippi': 'MS', 'Missississippi': 'MS',
    'Louisanna': 'LA', 'Lousiana': 'LA', 'Misourri': 'MO',
    'Flo': 'FL'
}

# Apply misspelling corrections
for pattern, state in misspelling_patterns.items():
    mask = swac_fb['hometown'].str.contains(pattern, na=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_applied += mask.sum()

print(f"   🔤 Misspelling fixes: {len(misspelling_patterns)} patterns applied")

# Additional pattern-based corrections for common issues
additional_patterns = {
    # Hometowns with specific misspellings or variations
    ', Ms': 'MS',  # Hometowns ending with ", Ms"
    'Ia.': 'IA',   # Iowa abbreviation variation
    'Fra.': 'FL',  # Florida misspelling
    'N.Y': 'NY',   # New York variation
}

# Apply additional pattern corrections
for pattern, state in additional_patterns.items():
    if pattern == ', Ms':
        mask = swac_fb['hometown'].str.contains(pattern, na=False) & swac_fb['player_state'].isnull()
    elif pattern == 'Ia.':
        mask = swac_fb['hometown'].str.endswith(pattern, na=False) & swac_fb['player_state'].isnull()
    elif pattern == 'Fra.':
        mask = swac_fb['hometown'].str.contains(pattern, na=False) & swac_fb['player_state'].isnull()
    else:
        mask = swac_fb['hometown'].str.endswith(pattern, na=False) & swac_fb['player_state'].isnull()
    
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_applied += mask.sum()

print(f"   ➕ Additional pattern fixes: {len(additional_patterns)} patterns applied")

# Special format corrections
special_fixes = [
    ('DentonTx', 'Denton, TX', 'TX'),
    ('Troy, Al', None, 'AL'),
    ('Natch', None, 'MS'),
    ('Hollywood, Fra.', 'Hollywood, FL', 'FL'),
    ('Tucker, Ga,', 'Tucker, GA', 'GA'),
    ('Greenlawn, N.Y', None, 'NY')
]

# Apply special format fixes
for pattern, new_hometown, state in special_fixes:
    mask = swac_fb['hometown'].str.contains(pattern, na=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        if new_hometown:
            swac_fb.loc[mask, 'hometown'] = new_hometown
        corrections_applied += mask.sum()

print(f"   🔧 Special format fixes: {len(special_fixes)} patterns applied")

# High school location patterns
hs_patterns = {
    '/ Homestead HS': ('Homestead, FL', 'FL'),
    '/ Plantation HS': ('Plantation, FL', 'FL'),
    '/ P.K. Yonge HS': ('Gainesville, FL', 'FL'),
    '/ Curie HS': ('Chicago, IL', 'IL'),
    '/ Archbishop Curley': ('Miami, FL', 'FL'),
    '/ Jefferson HS': ('Tampa, FL', 'FL'),
    '/ Norland HS': ('Miami, FL', 'FL'),
    '/ Parkway Academy': ('Miramar, FL', 'FL'),
    'Lake Highland Prep': ('Orlando, FL', 'FL')
}

# Apply high school patterns
for pattern, (city_state, state) in hs_patterns.items():
    mask = swac_fb['hometown'].str.contains(pattern, na=False, regex=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'hometown'] = city_state
        swac_fb.loc[mask, 'player_state'] = state
        corrections_applied += mask.sum()

print(f"   🏫 High school patterns: {len(hs_patterns)} patterns applied")

# Additional comprehensive corrections from your development work
print(f"   🔧 Applying additional comprehensive corrections...")

# Direct index-based corrections (from your most successful runs)
index_corrections = {
    2035: 'AL',   # Linden,Al
    2196: 'FL',   # Sunrise  
    3843: 'IL',
    11714: 'MI',
    15147: 'FL',
    15030: 'FL', 
    15256: 'FL',
    15556: 'NC'
}

# Apply index corrections if they exist and are still missing
for idx, state in index_corrections.items():
    if idx in swac_fb.index and pd.isna(swac_fb.loc[idx, 'player_state']):
        swac_fb.loc[idx, 'player_state'] = state
        corrections_applied += 1

print(f"   📍 Index-based corrections: {len(index_corrections)} specific indices targeted")

# Name-based corrections for players with specific issues
name_based_corrections = {
    'Kobe Love': 'TN',
    'Ebenezer Dibula': 'INTL',
}

# Apply name-based corrections
for name, state in name_based_corrections.items():
    mask = (swac_fb['name'] == name) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_applied += mask.sum()

print(f"   👤 Name-based corrections: {len(name_based_corrections)} specific names targeted")

# Comprehensive substring corrections
substring_corrections = {
    'Waterloo, Ia.': 'IA',
    ', AL': 'AL',
    ', LA': 'LA', 
    ', Ms': 'MS',
    'Fra.': 'FL',
    'N.Y': 'NY'
}

# Apply substring corrections
for substring, state in substring_corrections.items():
    mask = swac_fb['hometown'].str.contains(substring, na=False) & swac_fb['player_state'].isnull()
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        corrections_applied += mask.sum()

print(f"   🔍 Substring corrections: {len(substring_corrections)} patterns applied")

# Calculate progress after pattern corrections
missing_after_patterns = swac_fb['player_state'].isnull().sum()
success_rate_after_patterns = ((len(swac_fb) - missing_after_patterns) / len(swac_fb) * 100)

print(f"\n📊 Pattern corrections summary:")
print(f"   🔧 Total pattern corrections: {corrections_applied}")
print(f"   📉 Missing reduced: {baseline_missing} → {missing_after_patterns}")
print(f"   📈 Success rate: {baseline_success_rate:.2f}% → {success_rate_after_patterns:.2f}%")

pattern_missing = missing_after_patterns

🔧 Applying systematic pattern-based corrections...
   🏙️ City mappings: 24 patterns applied
   📝 Suffix corrections: 16 patterns applied
   🔤 Misspelling fixes: 7 patterns applied
   ➕ Additional pattern fixes: 4 patterns applied
   🔧 Special format fixes: 6 patterns applied
   🏫 High school patterns: 9 patterns applied
   🔧 Applying additional comprehensive corrections...
   📍 Index-based corrections: 8 specific indices targeted
   👤 Name-based corrections: 2 specific names targeted
   🔍 Substring corrections: 6 patterns applied

📊 Pattern corrections summary:
   🔧 Total pattern corrections: 165
   📉 Missing reduced: 325 → 160
   📈 Success rate: 97.91% → 98.97%
   📝 Suffix corrections: 16 patterns applied
   🔤 Misspelling fixes: 7 patterns applied
   ➕ Additional pattern fixes: 4 patterns applied
   🔧 Special format fixes: 6 patterns applied
   🏫 High school patterns: 9 patterns applied
   🔧 Applying additional comprehensive corrections...
   📍 Index-based corrections: 8 specific indi

In [21]:
# ===== SECTION 6: INDIVIDUAL PLAYER CORRECTIONS =====

print("👤 Applying individual player corrections...")

individual_corrections = 0

# High-priority individual player corrections
player_corrections = [
    # Name, Team, Season(s), Hometown, High School, Previous School, State
    ('Andre Washington', 'Alcorn State', None, 'Hopkins, SC', 'Ridgeview HS', 'UNC Charlotte', 'SC'),
    ('Alexander Shaw', 'Jackson State', None, 'Vicksburg, MS', None, None, 'MS'),
    ('Marquell Rozier', 'Bethune-Cookman', None, None, None, None, 'NC'),
    ('Tekeven Thomas', 'Bethune-Cookman', None, None, None, None, 'AL'),
    ('Brandon Duncan', 'UAPB', None, None, None, None, 'NY'),
    ('Austin Jones', 'Alabama A&M', None, None, None, None, 'IL'),
    ('Eric Smith', 'Florida A&M', None, 'Opa Locka, FL', 'Norland HS', None, 'FL'),
    ('Keir Abrams', 'Florida A&M', None, 'Oakland, CA', None, None, 'CA'),
    ('Juavon Brown', 'Jackson State', None, None, None, None, 'LA'),
    ('Devin Tribble', 'Alcorn State', None, None, None, None, 'MS'),
    ('Antonio Wells', 'Alcorn State', None, None, None, None, 'MS'),
    ('Keshuan Blackmon', 'Bethune-Cookman', None, None, None, None, 'MS'),
    ('Jamie Gleaton', 'Texas Southern', None, 'Batesburg-Leesville, SC', 'Batesburg-Leesville HS', None, 'SC'),
    
    # Round 2 individual corrections
    ('Bennie Peoples', 'Grambling', [2010, 2011], 'Vicksburg, MS', None, 'Co-Lin CC', 'MS'),
    ('Radonte Womack', 'Bethune-Cookman', [2021], None, None, None, 'MS'),
    ('DeAngelo Alexander', 'Prairie View A&M', [2022], None, None, None, 'TX'),
    ('Van, Phillips,', 'Grambling', [2011], 'Irondale, AL', 'Shades Valley HS', None, 'AL'),
    ('Marc Lucien', 'UAPB', [2015], None, None, None, 'NY'),
    ('Reggie Polite', 'Bethune-Cookman', [2013, 2014, 2015], 'Bartow, FL', None, None, 'FL'),
    ("Ja'sion Greathouse", 'Southern', [2022], None, None, None, 'IL'),
    ('Raequan Prince', 'UAPB', [2021], None, None, None, 'OH'),
    
    # Round 3 corrections
    ('Genoa Sartin', 'Alcorn State', [2014], 'Brookhaven, MS', 'Brookhaven HS', None, 'MS'),
    ('Devin Mitchell', 'UAPB', [2015], 'Perris, CA', None, None, 'CA'),
    ('Blain Winston', 'Southern', [2013], 'Monroe, LA', 'Richwood HS', 'UL-Lafayette', 'LA'),
    ('Jared Mitchell', 'Bethune-Cookman', [2013], 'Chatham, VA', None, 'Ole Miss', 'VA'),
    
    # Final round Bethune-Cookman corrections
    ('Maurice Roberts', 'Bethune-Cookman', [2013, 2014], 'Immokalee, FL', 'Immokalee HS', None, 'FL'),
    ('Johnathan Moment', 'Bethune-Cookman', [2013], 'Orlando, FL', 'Lake Highland Prep', None, 'FL'),
    ('Buddy Collins', 'Bethune-Cookman', [2011], 'DeLand, FL', None, None, 'FL'),
    ('Daniel Jackson', 'Bethune-Cookman', [2011], 'Bartow, FL', None, None, 'FL'),
    ('Tavaris Bell', 'Bethune-Cookman', [2011], 'Jacksonville, FL', None, None, 'FL'),
    ('Ryan Davis', 'Bethune-Cookman', [2011], 'Tampa, FL', None, None, 'FL'),
    ('Jens Howe', 'Bethune-Cookman', [2014], 'Manti, UT', None, None, 'UT'),
    ('Jawad Yatim', 'Bethune-Cookman', [2011], 'Shrewsbury, MA', None, None, 'MA'),
    ('Xavier Reese', 'Bethune-Cookman', [2011], None, None, None, 'FL'),
    ('Kyle Bailey', 'Bethune-Cookman', [2012], None, None, None, 'CA'),
    ('Maurice Francois', 'Bethune-Cookman', [2011], 'Palm Bay, FL', None, None, 'FL'),
    ('Jazz Moss', None, None, 'Ft. Lauderdale, FL', None, None, 'FL')
]

# Additional corrections I missed from your development work
additional_player_corrections = [
    # Add missing corrections that were in your original rounds
    ('Kobe Love', None, None, None, None, None, 'TN'),
    ('Ebenezer Dibula', None, None, None, None, None, 'INTL'),
    # Add more based on patterns you found
]

# Combine all player corrections
all_player_corrections = player_corrections + additional_player_corrections

# Apply individual player corrections
for player_data in player_corrections:
    name, team, seasons, hometown, high_school, previous_school, state = player_data
    
    # Build mask for this player
    mask = (swac_fb['name'] == name) & swac_fb['player_state'].isnull()
    
    if team:
        mask = mask & (swac_fb['team'] == team)
    
    if seasons:
        if isinstance(seasons, list):
            mask = mask & swac_fb['season'].isin(seasons)
        else:
            mask = mask & (swac_fb['season'] == seasons)
    
    # Apply corrections if mask matches any rows
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        if hometown:
            swac_fb.loc[mask, 'hometown'] = hometown
        if high_school:
            swac_fb.loc[mask, 'high_school'] = high_school
        if previous_school:
            swac_fb.loc[mask, 'previous_school'] = previous_school
        individual_corrections += mask.sum()

# Special team-based corrections
team_corrections = [
    # Grambling 2011 with "Hodge HS)" in previous_school
    ('Grambling', 2011, 'previous_school', 'Hodge HS)', 'Jonesboro, LA', 'Jonesboro Hodge HS', 'LA'),
    # Stephen names for Grambling 2010/2011
    ('Grambling', [2010, 2011], 'name_startswith', 'Stephen', 'Eight Mile, AL', 'McGill-Toolen HS', 'AL')
]

# Apply team-based corrections
for team_data in team_corrections:
    team, seasons, field, pattern, hometown, high_school, state = team_data
    
    # Build mask
    mask = (swac_fb['team'] == team) & swac_fb['player_state'].isnull()
    
    if isinstance(seasons, list):
        mask = mask & swac_fb['season'].isin(seasons)
    else:
        mask = mask & (swac_fb['season'] == seasons)
    
    if field == 'previous_school':
        mask = mask & swac_fb['previous_school'].str.contains(pattern, na=False, regex=False)
    elif field == 'name_startswith':
        mask = mask & swac_fb['name'].str.startswith(pattern, na=False)
    
    if mask.any():
        swac_fb.loc[mask, 'player_state'] = state
        swac_fb.loc[mask, 'hometown'] = hometown
        swac_fb.loc[mask, 'high_school'] = high_school
        individual_corrections += mask.sum()

# Calculate progress after individual corrections
missing_after_individual = swac_fb['player_state'].isnull().sum()
success_rate_after_individual = ((len(swac_fb) - missing_after_individual) / len(swac_fb) * 100)

print(f"\n👥 Individual corrections summary:")
print(f"   👤 Individual player fixes: {len(player_corrections)}")
print(f"   🏫 Team-based corrections: {len(team_corrections)}")
print(f"   ✅ Total individual corrections: {individual_corrections}")
print(f"   📉 Missing reduced: {pattern_missing} → {missing_after_individual}")
print(f"   📈 Success rate: {success_rate_after_patterns:.2f}% → {success_rate_after_individual:.2f}%")

👤 Applying individual player corrections...

👥 Individual corrections summary:
   👤 Individual player fixes: 37
   🏫 Team-based corrections: 2
   ✅ Total individual corrections: 30
   📉 Missing reduced: 160 → 130
   📈 Success rate: 98.97% → 99.17%


In [22]:
# ===== SECTION 7: DATA QUALITY ASSESSMENT AND VALIDATION =====

print("📊 Conducting comprehensive data quality assessment...")

# Final statistics
final_missing = swac_fb['player_state'].isnull().sum()
final_intl = (swac_fb['player_state'] == 'INTL').sum()
final_success_rate = ((len(swac_fb) - final_missing) / len(swac_fb) * 100)

# Progress summary
total_corrections = corrections_applied + individual_corrections
improvement = baseline_success_rate - final_success_rate

print(f"\n🎯 FINAL PROCESSING RESULTS:")
print(f"   📊 Total players processed: {len(swac_fb):,}")
print(f"   ✅ Successfully assigned states: {len(swac_fb) - final_missing:,}")
print(f"   🌍 International players: {final_intl}")
print(f"   ❌ Still missing: {final_missing}")
print(f"   🏆 FINAL SUCCESS RATE: {final_success_rate:.2f}%")

print(f"\n📈 IMPROVEMENT SUMMARY:")
print(f"   🚀 Starting success rate: {baseline_success_rate:.2f}%")
print(f"   🎯 Final success rate: {final_success_rate:.2f}%")
print(f"   📊 Total improvement: {final_success_rate - baseline_success_rate:.2f} percentage points")
print(f"   🔧 Total corrections applied: {total_corrections}")

# Success rate validation
target_rate = 99.36
if final_success_rate >= target_rate:
    print(f"\n🎉 SUCCESS! Target of {target_rate}% achieved!")
    print(f"   🏆 Exceeded target by {final_success_rate - target_rate:.2f} percentage points")
else:
    print(f"\n📊 Current rate: {final_success_rate:.2f}% (Target: {target_rate}%)")
    remaining_needed = int((target_rate/100 * len(swac_fb)) - (len(swac_fb) - final_missing))
    print(f"   🎯 Need {remaining_needed} more corrections to reach target")

# State distribution analysis
print(f"\n🗺️ TOP 10 STATES BY PLAYER COUNT:")
state_counts = swac_fb['player_state'].value_counts(dropna=False).head(10)
for i, (state, count) in enumerate(state_counts.items(), 1):
    percentage = (count / len(swac_fb)) * 100
    state_name = state if state in ['INTL', None] else f"{state}"
    print(f"   {i:2d}. {state_name}: {count:,} players ({percentage:.1f}%)")

# Team performance summary
print(f"\n🏫 TEAM STATE ASSIGNMENT PERFORMANCE:")
team_performance = []
for team in sorted(swac_fb['team'].unique()):
    team_data = swac_fb[swac_fb['team'] == team]
    team_missing = team_data['player_state'].isnull().sum()
    team_total = len(team_data)
    team_rate = ((team_total - team_missing) / team_total * 100) if team_total > 0 else 0
    team_performance.append((team, team_rate, team_missing, team_total))

# Sort by success rate (descending)
team_performance.sort(key=lambda x: x[1], reverse=True)
for team, rate, missing, total in team_performance:
    print(f"   {team:20s}: {rate:5.1f}% ({missing:2d} missing of {total:,})")

# Data quality metrics
print(f"\n📋 DATA QUALITY METRICS:")
print(f"   🏠 Complete hometown data: {swac_fb['hometown'].notna().sum():,} ({swac_fb['hometown'].notna().sum()/len(swac_fb)*100:.1f}%)")
print(f"   🎓 Complete high school data: {swac_fb['high_school'].notna().sum():,} ({swac_fb['high_school'].notna().sum()/len(swac_fb)*100:.1f}%)")
print(f"   📚 Complete previous school data: {swac_fb['previous_school'].notna().sum():,} ({swac_fb['previous_school'].notna().sum()/len(swac_fb)*100:.1f}%)")
print(f"   🗺️ Complete player state data: {swac_fb['player_state'].notna().sum():,} ({final_success_rate:.2f}%)")
print(f"   🏛️ Complete team state data: {swac_fb['team_state'].notna().sum():,} ({swac_fb['team_state'].notna().sum()/len(swac_fb)*100:.1f}%)")

# Display sample of final dataset
print(f"\n📋 FINAL DATASET SAMPLE:")
sample_df = swac_fb[['name', 'team', 'season', 'hometown', 'player_state', 'team_state', 'is_international']].head(10)
print(sample_df.to_string(index=False))

📊 Conducting comprehensive data quality assessment...

🎯 FINAL PROCESSING RESULTS:
   📊 Total players processed: 15,571
   ✅ Successfully assigned states: 15,441
   🌍 International players: 106
   ❌ Still missing: 130
   🏆 FINAL SUCCESS RATE: 99.17%

📈 IMPROVEMENT SUMMARY:
   🚀 Starting success rate: 97.91%
   🎯 Final success rate: 99.17%
   📊 Total improvement: 1.25 percentage points
   🔧 Total corrections applied: 195

📊 Current rate: 99.17% (Target: 99.36%)
   🎯 Need 30 more corrections to reach target

🗺️ TOP 10 STATES BY PLAYER COUNT:
    1. FL: 3,443 players (22.1%)
    2. TX: 2,657 players (17.1%)
    3. LA: 2,129 players (13.7%)
    4. AL: 1,531 players (9.8%)
    5. MS: 1,427 players (9.2%)
    6. GA: 1,224 players (7.9%)
    7. CA: 471 players (3.0%)
    8. TN: 380 players (2.4%)
    9. AR: 374 players (2.4%)
   10. IL: 223 players (1.4%)

🏫 TEAM STATE ASSIGNMENT PERFORMANCE:
   Mississippi Valley State: 100.0% ( 0 missing of 1,090)
   Jackson State       :  99.9% ( 1 missing

In [23]:
# ===== SECTION 8: FINAL DATASET VALIDATION AND EXPORT =====

print("📁 Finalizing cleaned dataset...")

# Create final clean dataset
final_dataset = swac_fb.copy()

# Ensure all required columns are present
required_columns = ['team', 'season', 'name', 'high_school', 'hometown', 'previous_school', 'class', 'team_state', 'player_state', 'is_international']
missing_columns = [col for col in required_columns if col not in final_dataset.columns]
if missing_columns:
    print(f"⚠️ Warning: Missing columns: {missing_columns}")
else:
    print(f"✅ All required columns present: {len(required_columns)} columns")

# Final missing players analysis (if any remain)
remaining_missing = final_dataset[final_dataset['player_state'].isnull()]
if len(remaining_missing) > 0:
    print(f"\n❌ REMAINING MISSING PLAYERS ({len(remaining_missing)} total):")
    missing_by_team = remaining_missing['team'].value_counts()
    for team, count in missing_by_team.items():
        total_team = len(final_dataset[final_dataset['team'] == team])
        pct = (count / total_team) * 100
        print(f"   {team}: {count} missing ({pct:.1f}% of {total_team} players)")
    
    # Show sample of remaining missing for potential future corrections
    print(f"\n📋 SAMPLE OF REMAINING MISSING:")
    sample_missing = remaining_missing[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].head(10)
    print(sample_missing.to_string(index=False))
else:
    print(f"\n🎉 PERFECT! ALL {len(final_dataset):,} PLAYERS HAVE STATE ASSIGNMENTS!")

# Data validation checks
print(f"\n🔍 DATA VALIDATION CHECKS:")
print(f"   ✅ Total records: {len(final_dataset):,}")
print(f"   ✅ No duplicate indices: {len(final_dataset) == len(final_dataset.index.unique())}")
print(f"   ✅ Valid seasons: {final_dataset['season'].min()}-{final_dataset['season'].max()}")
print(f"   ✅ Valid teams: {len(final_dataset['team'].unique())} teams")
print(f"   ✅ Player states assigned: {final_dataset['player_state'].notna().sum():,} ({final_success_rate:.2f}%)")

# Export options (uncomment as needed)
print(f"\n💾 EXPORT OPTIONS:")
print(f"   # Export to CSV:")
print(f"   # final_dataset.to_csv('SWAC_Football_Cleaned.csv', index=False)")
print(f"   # Export summary statistics:")
print(f"   # final_dataset.describe().to_csv('SWAC_Data_Summary.csv')")

# Final success message
print(f"\n" + "="*80)
print(f"🏆 SWAC FOOTBALL DATA CLEANING COMPLETE!")
print(f"="*80)
print(f"📊 Dataset: {len(final_dataset):,} players across {len(final_dataset['team'].unique())} teams")
print(f"📅 Coverage: {final_dataset['season'].min()}-{final_dataset['season'].max()} seasons")
print(f"🎯 Success Rate: {final_success_rate:.2f}%")
print(f"🌍 International Players: {final_intl}")
print(f"✅ Dataset cleaned and ready for analysis!")
print(f"="*80)

# Display final dataset structure and sample
print(f"\n📊 CLEANED DATASET STRUCTURE:")
print(f"Shape: {final_dataset.shape}")
print(f"Columns: {list(final_dataset.columns)}")
print(f"\nSample of cleaned data:")
final_dataset[['name', 'team', 'season', 'hometown', 'player_state', 'team_state', 'is_international']].head()

📁 Finalizing cleaned dataset...
✅ All required columns present: 10 columns

❌ REMAINING MISSING PLAYERS (130 total):
   Bethune-Cookman: 34 missing (2.3% of 1455 players)
   Alabama A&M: 24 missing (2.0% of 1203 players)
   Alabama State: 14 missing (0.9% of 1588 players)
   Southern: 14 missing (1.1% of 1313 players)
   Prairie View A&M: 12 missing (0.8% of 1481 players)
   Alcorn State: 11 missing (0.8% of 1295 players)
   Florida A&M: 7 missing (0.5% of 1508 players)
   UAPB: 7 missing (0.6% of 1171 players)
   Grambling: 4 missing (0.3% of 1387 players)
   Texas Southern: 2 missing (0.1% of 1334 players)
   Jackson State: 1 missing (0.1% of 746 players)

📋 SAMPLE OF REMAINING MISSING:
          name          team  season           hometown high_school                      previous_school
 Brandon McCoy Jackson State    2018 Memphis, Tenneesse         NaN                                 None
  Josiah Drain Alabama State    2024             Tacoma         NaN Central Wasington - Okla

,name,team,season,hometown,player_state,team_state,is_international
0,Travis Terrell Jr.,Jackson State,2025,"Atlanta, Ga.",GA,MS,False
1,Jeremiah Williams,Jackson State,2025,"Lexington, Miss.",MS,MS,False
2,Khamauri Rogers,Jackson State,2025,"Madison, Miss.",MS,MS,False
3,Shemar Savage,Jackson State,2025,"Lompoc, Calif.",CA,MS,False
4,Ja'Naylon Dupree,Jackson State,2025,"Philadelphia, Miss.",MS,MS,False


In [24]:
# ===== DETAILED ANALYSIS OF REMAINING MISSING PLAYERS =====

print("🔍 DETAILED ANALYSIS OF REMAINING 130 MISSING PLAYERS")
print("=" * 60)

# Get all missing players
missing_players = swac_fb[swac_fb['player_state'].isnull()].copy()

print(f"📊 Current Status:")
print(f"   Total players: {len(swac_fb):,}")
print(f"   Successfully assigned: {len(swac_fb) - len(missing_players):,} (99.17%)")
print(f"   Still missing: {len(missing_players)} (0.83%)")
print(f"   Target: 99.36% (need {30} more corrections)")

print(f"\n🏫 Missing Players by Team:")
missing_by_team = missing_players['team'].value_counts()
for team, count in missing_by_team.items():
    print(f"   {team:20s}: {count:2d} missing")

print(f"\n📅 Missing Players by Season:")
missing_by_season = missing_players['season'].value_counts().sort_index()
print(missing_by_season.to_string())

print(f"\n🔍 Sample of Missing Players with Hometowns:")
sample_missing = missing_players[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].head(20)
print(sample_missing.to_string(index=False))

print(f"\n🏠 Missing Players with Hometown Data:")
missing_with_hometown = missing_players[missing_players['hometown'].notna()]
print(f"   Players with hometown: {len(missing_with_hometown)} out of {len(missing_players)}")

if len(missing_with_hometown) > 0:
    print(f"\n   Top hometown patterns in missing data:")
    hometown_patterns = missing_with_hometown['hometown'].value_counts().head(10)
    print(hometown_patterns.to_string())
    
    print(f"\n   Sample missing with hometowns:")
    print(missing_with_hometown[['name', 'team', 'hometown']].head(15).to_string(index=False))

print(f"\n📚 Missing Players with Previous School Data:")
missing_with_prev_school = missing_players[missing_players['previous_school'].notna()]
print(f"   Players with previous school: {len(missing_with_prev_school)} out of {len(missing_players)}")

if len(missing_with_prev_school) > 0:
    print(f"\n   Sample missing with previous school:")
    print(missing_with_prev_school[['name', 'team', 'previous_school']].head(10).to_string(index=False))

🔍 DETAILED ANALYSIS OF REMAINING 130 MISSING PLAYERS
📊 Current Status:
   Total players: 15,571
   Successfully assigned: 15,441 (99.17%)
   Still missing: 130 (0.83%)
   Target: 99.36% (need 30 more corrections)

🏫 Missing Players by Team:
   Bethune-Cookman     : 34 missing
   Alabama A&M         : 24 missing
   Alabama State       : 14 missing
   Southern            : 14 missing
   Prairie View A&M    : 12 missing
   Alcorn State        : 11 missing
   Florida A&M         :  7 missing
   UAPB                :  7 missing
   Grambling           :  4 missing
   Texas Southern      :  2 missing
   Jackson State       :  1 missing

📅 Missing Players by Season:
season
2010     2
2011    16
2012     7
2013    21
2014     8
2015    13
2016     4
2017     3
2018     4
2020     7
2021    10
2022    10
2023     9
2024    11
2025     5

🔍 Sample of Missing Players with Hometowns:
             name          team  season           hometown     high_school                      previous_school
    

In [25]:
# ===== ANALYSIS: EASIEST CORRECTIONS TO MAKE =====

print("\n🎯 TARGETED CORRECTIONS TO REACH 99.36% TARGET")
print("=" * 50)

print("🔧 IMMEDIATE CORRECTIONS IDENTIFIED:")

corrections_found = 0

# 1. Memphis, Tenneesse -> TN (obvious typo)
memphis_player = swac_fb[(swac_fb['hometown'] == 'Memphis, Tenneesse') & swac_fb['player_state'].isnull()]
if len(memphis_player) > 0:
    print(f"   1. Memphis, Tenneesse → TN (typo): {len(memphis_player)} player(s)")
    corrections_found += len(memphis_player)

# 2. Tacoma -> WA (Washington state)
tacoma_player = swac_fb[(swac_fb['hometown'] == 'Tacoma') & swac_fb['player_state'].isnull()]
if len(tacoma_player) > 0:
    print(f"   2. Tacoma → WA: {len(tacoma_player)} player(s)")
    corrections_found += len(tacoma_player)

# 3. Prattville -> AL (Alabama city)
prattville_player = swac_fb[(swac_fb['hometown'] == 'Prattville') & swac_fb['player_state'].isnull()]
if len(prattville_player) > 0:
    print(f"   3. Prattville → AL: {len(prattville_player)} player(s)")
    corrections_found += len(prattville_player)

# 4. St. Paul, MN. -> MN (Minnesota)
stpaul_players = swac_fb[(swac_fb['hometown'] == 'St. Paul, MN.') & swac_fb['player_state'].isnull()]
if len(stpaul_players) > 0:
    print(f"   4. St. Paul, MN. → MN: {len(stpaul_players)} player(s)")
    corrections_found += len(stpaul_players)

# 5. Houston -> TX (Texas city)
houston_players = swac_fb[(swac_fb['hometown'] == 'Houston') & swac_fb['player_state'].isnull()]
if len(houston_players) > 0:
    print(f"   5. Houston → TX: {len(houston_players)} player(s)")
    corrections_found += len(houston_players)

# 6. Conway, AR. -> AR (Arkansas)
conway_players = swac_fb[(swac_fb['hometown'] == 'Conway, AR.') & swac_fb['player_state'].isnull()]
if len(conway_players) > 0:
    print(f"   6. Conway, AR. → AR: {len(conway_players)} player(s)")
    corrections_found += len(conway_players)

# 7. Pine Bluff,AR. -> AR (Arkansas)
pine_bluff_players = swac_fb[(swac_fb['hometown'] == 'Pine Bluff,AR.') & swac_fb['player_state'].isnull()]
if len(pine_bluff_players) > 0:
    print(f"   7. Pine Bluff,AR. → AR: {len(pine_bluff_players)} player(s)")
    corrections_found += len(pine_bluff_players)

# 8. Ft. Lauderdale -> FL (Florida)
ftlaud_players = swac_fb[(swac_fb['hometown'] == 'Ft. Lauderdale') & swac_fb['player_state'].isnull()]
if len(ftlaud_players) > 0:
    print(f"   8. Ft. Lauderdale → FL: {len(ftlaud_players)} player(s)")
    corrections_found += len(ftlaud_players)

# 9. High school patterns (likely Florida)
raines_players = swac_fb[(swac_fb['hometown'] == '/ Raines HS') & swac_fb['player_state'].isnull()]
if len(raines_players) > 0:
    print(f"   9. / Raines HS → FL: {len(raines_players)} player(s)")
    corrections_found += len(raines_players)

armwood_players = swac_fb[(swac_fb['hometown'] == '/ Armwood HS') & swac_fb['player_state'].isnull()]
if len(armwood_players) > 0:
    print(f"  10. / Armwood HS → FL: {len(armwood_players)} player(s)")
    corrections_found += len(armwood_players)

uchs_players = swac_fb[(swac_fb['hometown'] == 'University Christian HS') & swac_fb['player_state'].isnull()]
if len(uchs_players) > 0:
    print(f"  11. University Christian HS → FL: {len(uchs_players)} player(s)")
    corrections_found += len(uchs_players)

ecglass_players = swac_fb[(swac_fb['hometown'] == '/ E.C. Glass HS') & swac_fb['player_state'].isnull()]
if len(ecglass_players) > 0:
    print(f"  12. / E.C. Glass HS → VA: {len(ecglass_players)} player(s)")
    corrections_found += len(ecglass_players)

# 10. American Samoa
olosega_players = swac_fb[(swac_fb['hometown'] == 'Olosega') & swac_fb['player_state'].isnull()]
if len(olosega_players) > 0:
    print(f"  13. Olosega → INTL (American Samoa): {len(olosega_players)} player(s)")
    corrections_found += len(olosega_players)

print(f"\n📊 SUMMARY:")
print(f"   Easy corrections identified: {corrections_found}")
print(f"   Target needed: 30")
print(f"   {'✅ SUFFICIENT!' if corrections_found >= 30 else '⚠️ Need more analysis'}")

if corrections_found >= 30:
    print(f"\n🎉 Great! We have {corrections_found} easy corrections identified.")
    print(f"   This will easily get us to the 99.36% target!")
else:
    print(f"\n🔍 Need to find {30 - corrections_found} more corrections.")
    print("   Let's look at team-specific patterns or previous school data...")

# Show details of the easy corrections
print(f"\n📋 DETAILED VIEW OF EASIEST CORRECTIONS:")
easy_corrections = [
    ('Memphis, Tenneesse', 'TN'),
    ('Tacoma', 'WA'),
    ('Prattville', 'AL'),
    ('St. Paul, MN.', 'MN'),
    ('Houston', 'TX'),
    ('Conway, AR.', 'AR'),
    ('Pine Bluff,AR.', 'AR'),
    ('Ft. Lauderdale', 'FL'),
    ('/ Raines HS', 'FL'),
    ('/ Armwood HS', 'FL'),
    ('University Christian HS', 'FL'),
    ('/ E.C. Glass HS', 'VA'),
    ('Olosega', 'INTL')
]

for hometown, state in easy_corrections:
    matching = swac_fb[(swac_fb['hometown'] == hometown) & swac_fb['player_state'].isnull()]
    if len(matching) > 0:
        print(f"\n{hometown} → {state}:")
        for idx, row in matching.iterrows():
            print(f"   {row['name']} ({row['team']}, {row['season']})")


🎯 TARGETED CORRECTIONS TO REACH 99.36% TARGET
🔧 IMMEDIATE CORRECTIONS IDENTIFIED:
   1. Memphis, Tenneesse → TN (typo): 1 player(s)
   2. Tacoma → WA: 1 player(s)
   3. Prattville → AL: 1 player(s)
   4. St. Paul, MN. → MN: 3 player(s)
   5. Houston → TX: 7 player(s)
   6. Conway, AR. → AR: 2 player(s)
   7. Pine Bluff,AR. → AR: 2 player(s)
   8. Ft. Lauderdale → FL: 2 player(s)
   9. / Raines HS → FL: 5 player(s)
  10. / Armwood HS → FL: 3 player(s)
  11. University Christian HS → FL: 4 player(s)
  12. / E.C. Glass HS → VA: 3 player(s)
  13. Olosega → INTL (American Samoa): 3 player(s)

📊 SUMMARY:
   Easy corrections identified: 37
   Target needed: 30
   ✅ SUFFICIENT!

🎉 Great! We have 37 easy corrections identified.
   This will easily get us to the 99.36% target!

📋 DETAILED VIEW OF EASIEST CORRECTIONS:

Memphis, Tenneesse → TN:
   Brandon McCoy (Jackson State, 2018)

Tacoma → WA:
   Josiah Drain (Alabama State, 2024)

Prattville → AL:
   Earl Lucus Jr. (Alabama State, 2013)

St. 

In [26]:
# ===== SECTION 9: TARGETED CORRECTIONS TO REACH 99.36% =====

print("🎯 APPLYING TARGETED CORRECTIONS TO REACH 99.36% TARGET")
print("=" * 60)

targeted_corrections = 0
starting_missing = swac_fb['player_state'].isnull().sum()

# Apply the 37 easy corrections identified above
easy_corrections = [
    ('Memphis, Tenneesse', 'TN'),
    ('Tacoma', 'WA'), 
    ('Prattville', 'AL'),
    ('St. Paul, MN.', 'MN'),
    ('Houston', 'TX'),
    ('Conway, AR.', 'AR'),
    ('Pine Bluff,AR.', 'AR'),
    ('Ft. Lauderdale', 'FL'),
    ('/ Raines HS', 'FL'),
    ('/ Armwood HS', 'FL'),
    ('University Christian HS', 'FL'),
    ('/ E.C. Glass HS', 'VA'),
    ('Olosega', 'INTL')
]

print("🔧 Applying targeted corrections:")

for hometown, state in easy_corrections:
    mask = (swac_fb['hometown'] == hometown) & swac_fb['player_state'].isnull()
    count = mask.sum()
    if count > 0:
        swac_fb.loc[mask, 'player_state'] = state
        targeted_corrections += count
        print(f"   ✅ {hometown} → {state}: {count} correction(s)")

# Calculate final results
final_missing = swac_fb['player_state'].isnull().sum()
final_success_rate = ((len(swac_fb) - final_missing) / len(swac_fb) * 100)
target_rate = 99.36

print(f"\n📊 TARGETED CORRECTION RESULTS:")
print(f"   🔧 Targeted corrections applied: {targeted_corrections}")
print(f"   📉 Missing reduced: {starting_missing} → {final_missing}")
print(f"   📈 Success rate: {final_success_rate:.2f}%")
print(f"   🎯 Target rate: {target_rate:.2f}%")

if final_success_rate >= target_rate:
    print(f"\n🎉 SUCCESS! TARGET ACHIEVED!")
    print(f"   🏆 Exceeded target by {final_success_rate - target_rate:.2f} percentage points")
    print(f"   ✅ Dataset now has {final_success_rate:.2f}% completion rate")
else:
    remaining_needed = int((target_rate/100 * len(swac_fb)) - (len(swac_fb) - final_missing))
    print(f"\n📊 Progress made but still need {remaining_needed} more corrections")

print(f"\n🏫 UPDATED TEAM PERFORMANCE:")
team_performance = []
for team in sorted(swac_fb['team'].unique()):
    team_data = swac_fb[swac_fb['team'] == team]
    team_missing = team_data['player_state'].isnull().sum()
    team_total = len(team_data)
    team_rate = ((team_total - team_missing) / team_total * 100) if team_total > 0 else 0
    team_performance.append((team, team_rate, team_missing, team_total))

# Sort by success rate (descending)
team_performance.sort(key=lambda x: x[1], reverse=True)
for team, rate, missing, total in team_performance:
    print(f"   {team:20s}: {rate:5.1f}% ({missing:2d} missing of {total:,})")

# Show remaining missing count
remaining_missing = swac_fb[swac_fb['player_state'].isnull()]
print(f"\n📋 REMAINING MISSING PLAYERS: {len(remaining_missing)} total")
if len(remaining_missing) > 0:
    remaining_by_team = remaining_missing['team'].value_counts()
    for team, count in remaining_by_team.items():
        print(f"   {team}: {count} missing")

🎯 APPLYING TARGETED CORRECTIONS TO REACH 99.36% TARGET
🔧 Applying targeted corrections:
   ✅ Memphis, Tenneesse → TN: 1 correction(s)
   ✅ Tacoma → WA: 1 correction(s)
   ✅ Prattville → AL: 1 correction(s)
   ✅ St. Paul, MN. → MN: 3 correction(s)
   ✅ Houston → TX: 7 correction(s)
   ✅ Conway, AR. → AR: 2 correction(s)
   ✅ Pine Bluff,AR. → AR: 2 correction(s)
   ✅ Ft. Lauderdale → FL: 2 correction(s)
   ✅ / Raines HS → FL: 5 correction(s)
   ✅ / Armwood HS → FL: 3 correction(s)
   ✅ University Christian HS → FL: 4 correction(s)
   ✅ / E.C. Glass HS → VA: 3 correction(s)
   ✅ Olosega → INTL: 3 correction(s)

📊 TARGETED CORRECTION RESULTS:
   🔧 Targeted corrections applied: 37
   📉 Missing reduced: 130 → 93
   📈 Success rate: 99.40%
   🎯 Target rate: 99.36%

🎉 SUCCESS! TARGET ACHIEVED!
   🏆 Exceeded target by 0.04 percentage points
   ✅ Dataset now has 99.40% completion rate

🏫 UPDATED TEAM PERFORMANCE:
   Jackson State       : 100.0% ( 0 missing of 746)
   Mississippi Valley State: 100

In [28]:
# ===== SECTION 10: FINAL SPECIFIC CORRECTIONS =====

print("🎯 APPLYING FINAL SPECIFIC CORRECTIONS")
print("=" * 50)

final_corrections = 0
starting_missing = swac_fb['player_state'].isnull().sum()

print("🔧 Applying user-identified corrections:")

# 1. Grant Ewell Jr. (UAPB) - Oregon (hometown ends in "OR.")
grant_mask = (
    (swac_fb['name'] == 'Grant Ewell Jr') & 
    (swac_fb['team'] == 'UAPB') & 
    swac_fb['season'].isin([2022, 2023]) & 
    swac_fb['player_state'].isnull()
)
if grant_mask.any():
    swac_fb.loc[grant_mask, 'player_state'] = 'OR'
    final_corrections += grant_mask.sum()
    print(f"   ✅ Grant Ewell Jr. (UAPB) → OR: {grant_mask.sum()} correction(s)")

# 2. Felando Warr (Alcorn State, 2013) - Tennessee
felando_mask = (
    (swac_fb['name'] == 'Felando Warr') & 
    (swac_fb['team'] == 'Alcorn State') & 
    (swac_fb['season'] == 2013) & 
    swac_fb['player_state'].isnull()
)
if felando_mask.any():
    swac_fb.loc[felando_mask, 'player_state'] = 'TN'
    final_corrections += felando_mask.sum()
    print(f"   ✅ Felando Warr (Alcorn State) → TN: {felando_mask.sum()} correction(s)")

# 3. Bethune-Cookman 2013 high school format corrections
bethune_2013_hs_corrections = [
    ('/ Mandarin HS', 'FL'),      # Jacksonville, FL
    ('/ Myers Park HS', 'NC'),    # Charlotte, NC  
    ('/ Blanche Ely HS', 'FL'),   # Pompano Beach, FL
    ('/ Buchholz HS', 'FL'),      # Gainesville, FL
    ('/ Middleton HS', 'FL')      # Tampa, FL
]

for hs_pattern, state in bethune_2013_hs_corrections:
    # Check hometown field for this pattern
    hometown_mask = (
        (swac_fb['hometown'] == hs_pattern) & 
        (swac_fb['team'] == 'Bethune-Cookman') & 
        (swac_fb['season'] == 2013) & 
        swac_fb['player_state'].isnull()
    )
    
    # Also check high_school field for this pattern (without the "/ " prefix)
    hs_clean = hs_pattern.replace('/ ', '')
    hs_mask = (
        (swac_fb['high_school'] == hs_clean) & 
        (swac_fb['team'] == 'Bethune-Cookman') & 
        (swac_fb['season'] == 2013) & 
        swac_fb['player_state'].isnull()
    )
    
    combined_mask = hometown_mask | hs_mask
    
    if combined_mask.any():
        swac_fb.loc[combined_mask, 'player_state'] = state
        final_corrections += combined_mask.sum()
        print(f"   ✅ {hs_pattern} (Bethune-Cookman 2013) → {state}: {combined_mask.sum()} correction(s)")

# 4. DeLand HS correction for Bethune-Cookman
deland_mask = (
    (swac_fb['high_school'] == 'DeLand HS') & 
    (swac_fb['team'] == 'Bethune-Cookman') & 
    swac_fb['player_state'].isnull()
)
if deland_mask.any():
    swac_fb.loc[deland_mask, 'player_state'] = 'FL'
    final_corrections += deland_mask.sum()
    print(f"   ✅ DeLand HS (Bethune-Cookman) → FL: {deland_mask.sum()} correction(s)")

# 5. Additional pattern check for OR. endings (in case we missed any)
or_ending_mask = (
    swac_fb['hometown'].str.endswith('OR.', na=False) & 
    swac_fb['player_state'].isnull()
)
if or_ending_mask.any():
    swac_fb.loc[or_ending_mask, 'player_state'] = 'OR'
    final_corrections += or_ending_mask.sum()
    print(f"   ✅ Hometowns ending in 'OR.' → OR: {or_ending_mask.sum()} correction(s)")

# Calculate results
final_missing = swac_fb['player_state'].isnull().sum()
final_success_rate = ((len(swac_fb) - final_missing) / len(swac_fb) * 100)

print(f"\n📊 FINAL SPECIFIC CORRECTION RESULTS:")
print(f"   🔧 Final corrections applied: {final_corrections}")
print(f"   📉 Missing reduced: {starting_missing} → {final_missing}")
print(f"   📈 NEW Success rate: {final_success_rate:.2f}%")

if final_corrections > 0:
    print(f"\n🎉 Great! Applied {final_corrections} additional corrections!")
    print(f"   📈 Success rate improved to {final_success_rate:.2f}%")
    
    # Show who we corrected
    print(f"\n📋 Details of corrections made:")
    if grant_mask.any():
        grant_players = swac_fb[grant_mask]
        for _, player in grant_players.iterrows():
            print(f"   - {player['name']} ({player['team']}, {player['season']}) → OR")
    
    if felando_mask.any():
        felando_players = swac_fb[felando_mask]
        for _, player in felando_players.iterrows():
            print(f"   - {player['name']} ({player['team']}, {player['season']}) → TN")

# Show remaining count
remaining = swac_fb[swac_fb['player_state'].isnull()]
print(f"\n📋 REMAINING MISSING: {len(remaining)} players ({((len(remaining)/len(swac_fb))*100):.2f}%)")

if len(remaining) > 0:
    remaining_by_team = remaining['team'].value_counts()
    print("\n🏫 Remaining missing by team:")
    for team, count in remaining_by_team.items():
        print(f"   {team}: {count} missing")

🎯 APPLYING FINAL SPECIFIC CORRECTIONS
🔧 Applying user-identified corrections:
   ✅ Grant Ewell Jr. (UAPB) → OR: 2 correction(s)
   ✅ Felando Warr (Alcorn State) → TN: 1 correction(s)
   ✅ / Mandarin HS (Bethune-Cookman 2013) → FL: 1 correction(s)
   ✅ / Myers Park HS (Bethune-Cookman 2013) → NC: 1 correction(s)
   ✅ / Blanche Ely HS (Bethune-Cookman 2013) → FL: 1 correction(s)
   ✅ / Buchholz HS (Bethune-Cookman 2013) → FL: 1 correction(s)
   ✅ / Middleton HS (Bethune-Cookman 2013) → FL: 1 correction(s)
   ✅ DeLand HS (Bethune-Cookman) → FL: 1 correction(s)

📊 FINAL SPECIFIC CORRECTION RESULTS:
   🔧 Final corrections applied: 9
   📉 Missing reduced: 93 → 84
   📈 NEW Success rate: 99.46%

🎉 Great! Applied 9 additional corrections!
   📈 Success rate improved to 99.46%

📋 Details of corrections made:
   - Grant Ewell Jr (UAPB, 2023) → OR
   - Grant Ewell Jr (UAPB, 2022) → OR
   - Felando Warr (Alcorn State, 2013) → TN

📋 REMAINING MISSING: 84 players (0.54%)

🏫 Remaining missing by team:


In [29]:
# ===== VIEW REMAINING 84 MISSING PLAYERS =====

print("📋 REMAINING 84 MISSING PLAYERS - DETAILED VIEW")
print("=" * 60)

# Get all remaining missing players
remaining_missing = swac_fb[swac_fb['player_state'].isnull()].copy()

print(f"🎯 FANTASTIC PROGRESS!")
print(f"   Current success rate: {((len(swac_fb) - len(remaining_missing)) / len(swac_fb) * 100):.2f}%")
print(f"   Total remaining missing: {len(remaining_missing)} players")
print(f"   We've gone from 325 → {len(remaining_missing)} missing players!")

# Create a clean view for analysis
missing_df = remaining_missing[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].copy()

# Sort by team and season for easier viewing
missing_df = missing_df.sort_values(['team', 'season', 'name'])

print(f"\n🏫 Breakdown by team:")
team_counts = missing_df['team'].value_counts()
for team, count in team_counts.items():
    total_team = len(swac_fb[swac_fb['team'] == team])
    pct = (count / total_team) * 100
    print(f"   {team:20s}: {count:2d} missing ({pct:.1f}%)")

# Show a sample of what's left to help identify more patterns
print(f"\n📊 SAMPLE OF REMAINING MISSING PLAYERS:")
print("=" * 120)

# Show a focused sample
sample_size = min(30, len(missing_df))
sample_df = missing_df.head(sample_size)
print(f"Showing first {sample_size} of {len(missing_df)} remaining:")
print()

# Display in a more readable format
for idx, row in sample_df.iterrows():
    hometown_str = f"'{row['hometown']}'" if pd.notna(row['hometown']) and row['hometown'] != '' else "None"
    hs_str = f"'{row['high_school']}'" if pd.notna(row['high_school']) and row['high_school'] != '' else "None"
    prev_str = f"'{row['previous_school']}'" if pd.notna(row['previous_school']) and row['previous_school'] != '' else "None"
    
    print(f"{row['name']:<25} | {row['team']:<20} | {row['season']} | H: {hometown_str:<20} | HS: {hs_str:<20}")

print(f"\n... and {len(missing_df) - sample_size} more players")

# Quick analysis of patterns in remaining data
print(f"\n🔍 PATTERNS IN REMAINING DATA:")

# Check for any obvious patterns we might have missed
remaining_with_hometown = missing_df[missing_df['hometown'].notna() & (missing_df['hometown'] != '')]
if len(remaining_with_hometown) > 0:
    print(f"   Players with hometown data: {len(remaining_with_hometown)}")
    hometown_patterns = remaining_with_hometown['hometown'].value_counts().head(10)
    if len(hometown_patterns) > 0:
        print("   Top hometown patterns:")
        for hometown, count in hometown_patterns.items():
            print(f"      '{hometown}': {count} player(s)")

# Check high school patterns
remaining_with_hs = missing_df[missing_df['high_school'].notna() & (missing_df['high_school'] != '')]
if len(remaining_with_hs) > 0:
    print(f"\n   Players with high school data: {len(remaining_with_hs)}")
    hs_patterns = remaining_with_hs['high_school'].value_counts().head(5)
    if len(hs_patterns) > 0:
        print("   Top high school patterns:")
        for hs, count in hs_patterns.items():
            print(f"      '{hs}': {count} player(s)")

print(f"\n📊 FULL REMAINING DATASET:")
print("=" * 120)
missing_df

📋 REMAINING 84 MISSING PLAYERS - DETAILED VIEW
🎯 FANTASTIC PROGRESS!
   Current success rate: 99.46%
   Total remaining missing: 84 players
   We've gone from 325 → 84 missing players!

🏫 Breakdown by team:
   Alabama A&M         : 24 missing (2.0%)
   Southern            : 14 missing (1.1%)
   Bethune-Cookman     : 11 missing (0.8%)
   Alabama State       :  9 missing (0.6%)
   Florida A&M         :  7 missing (0.5%)
   Alcorn State        :  7 missing (0.5%)
   Prairie View A&M    :  5 missing (0.3%)
   Grambling           :  4 missing (0.3%)
   Texas Southern      :  2 missing (0.1%)
   UAPB                :  1 missing (0.1%)

📊 SAMPLE OF REMAINING MISSING PLAYERS:
Showing first 30 of 84 remaining:

Averee Giles              | Alabama A&M          | 2014 | H: None                 | HS: None                
Conard Johnson            | Alabama A&M          | 2014 | H: None                 | HS: None                
Kalian Jackson            | Alabama A&M          | 2014 | H: None     

,name,team,season,hometown,high_school,previous_school
3393,Averee Giles,Alabama A&M,2014,NaN,NaN,None
3365,Conard Johnson,Alabama A&M,2014,NaN,NaN,None
3394,Kalian Jackson,Alabama A&M,2014,NaN,NaN,None
3353,Lorenzo Jackson,Alabama A&M,2014,NaN,NaN,None
3104,Denzel Davis,Alabama A&M,2017,NaN,NaN,None
...,...,...,...,...,...,...
3697,Joseph Gonzales,Southern,2024,None,NaN,None
3587,Maverick Harrington,Southern,2025,None,NaN,None
7662,Costley Williams,Texas Southern,2010,None,NaN,None
7605,Jarbar Perkins,Texas Southern,2010,None,NaN,None


In [30]:
# ===== DIRECT FIX FOR FELANDO WARR =====

print("🔧 FIXING FELANDO WARR DIRECTLY")
print("=" * 40)

# Check Felando Warr's current status
felando_check = swac_fb[swac_fb['name'] == 'Felando Warr']
print("Current Felando Warr data:")
print(felando_check[['name', 'team', 'season', 'hometown', 'player_state']].to_string(index=True))

# Direct correction by index if needed
if 9919 in swac_fb.index and pd.isna(swac_fb.loc[9919, 'player_state']):
    print(f"\n🎯 Found Felando Warr at index 9919 - applying correction")
    swac_fb.loc[9919, 'player_state'] = 'TN'
    print(f"   ✅ Set index 9919 player_state to TN")
else:
    print(f"\n⚠️ Index 9919 either doesn't exist or already has a state assigned")

# Also apply by name/team/season mask as backup
felando_mask = (
    (swac_fb['name'] == 'Felando Warr') & 
    swac_fb['player_state'].isnull()
)

if felando_mask.any():
    swac_fb.loc[felando_mask, 'player_state'] = 'TN'
    print(f"   ✅ Applied correction via name mask: {felando_mask.sum()} correction(s)")
else:
    print(f"   ℹ️ No missing Felando Warr entries found via name mask")

# Verify the fix
felando_after = swac_fb[swac_fb['name'] == 'Felando Warr']
print(f"\nFelando Warr after correction:")
print(felando_after[['name', 'team', 'season', 'hometown', 'player_state']].to_string(index=True))

# Update our counts
remaining_after_fix = swac_fb[swac_fb['player_state'].isnull()]
new_success_rate = ((len(swac_fb) - len(remaining_after_fix)) / len(swac_fb) * 100)

print(f"\n📊 UPDATED COUNTS:")
print(f"   Remaining missing: {len(remaining_after_fix)} players")
print(f"   Success rate: {new_success_rate:.2f}%")

🔧 FIXING FELANDO WARR DIRECTLY
Current Felando Warr data:
              name          team  season     hometown player_state
9919  Felando Warr  Alcorn State    2013      Memphis           TN
9996  Felando Warr  Alcorn State    2012  Memphis, TN           TN

⚠️ Index 9919 either doesn't exist or already has a state assigned
   ℹ️ No missing Felando Warr entries found via name mask

Felando Warr after correction:
              name          team  season     hometown player_state
9919  Felando Warr  Alcorn State    2013      Memphis           TN
9996  Felando Warr  Alcorn State    2012  Memphis, TN           TN

📊 UPDATED COUNTS:
   Remaining missing: 84 players
   Success rate: 99.46%


In [31]:
# ===== FINAL VIEW OF REMAINING 84 MISSING PLAYERS =====

print("📋 CURRENT REMAINING 84 MISSING PLAYERS")
print("=" * 50)

# Get all remaining missing players
current_missing = swac_fb[swac_fb['player_state'].isnull()].copy()

print(f"🎯 EXCELLENT PROGRESS!")
print(f"   Success rate: {((len(swac_fb) - len(current_missing)) / len(swac_fb) * 100):.2f}%")
print(f"   Remaining missing: {len(current_missing)} players")

# Create the dataframe for viewing
missing_for_view = current_missing[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].copy()
missing_for_view = missing_for_view.sort_values(['team', 'season', 'name'])

print(f"\n🏫 By team:")
team_breakdown = missing_for_view['team'].value_counts()
for team, count in team_breakdown.items():
    print(f"   {team}: {count}")

print(f"\n📊 COMPLETE LIST OF REMAINING MISSING PLAYERS:")
print("="*100)

# Display the dataframe
missing_for_view

📋 CURRENT REMAINING 84 MISSING PLAYERS
🎯 EXCELLENT PROGRESS!
   Success rate: 99.46%
   Remaining missing: 84 players

🏫 By team:
   Alabama A&M: 24
   Southern: 14
   Bethune-Cookman: 11
   Alabama State: 9
   Florida A&M: 7
   Alcorn State: 7
   Prairie View A&M: 5
   Grambling: 4
   Texas Southern: 2
   UAPB: 1

📊 COMPLETE LIST OF REMAINING MISSING PLAYERS:


,name,team,season,hometown,high_school,previous_school
3393,Averee Giles,Alabama A&M,2014,NaN,NaN,None
3365,Conard Johnson,Alabama A&M,2014,NaN,NaN,None
3394,Kalian Jackson,Alabama A&M,2014,NaN,NaN,None
3353,Lorenzo Jackson,Alabama A&M,2014,NaN,NaN,None
3104,Denzel Davis,Alabama A&M,2017,NaN,NaN,None
...,...,...,...,...,...,...
3697,Joseph Gonzales,Southern,2024,None,NaN,None
3587,Maverick Harrington,Southern,2025,None,NaN,None
7662,Costley Williams,Texas Southern,2010,None,NaN,None
7605,Jarbar Perkins,Texas Southern,2010,None,NaN,None


In [32]:
# ===== EXPORT CLEANED DATASET =====

import os

print("💾 EXPORTING CLEANED SWAC DATASET")
print("=" * 40)

# Define export path and filename
export_dir = r"C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2"
export_filename = "SWAC_rosters_clean.csv"
export_path = os.path.join(export_dir, export_filename)

# Create directory if it doesn't exist
os.makedirs(export_dir, exist_ok=True)
print(f"📁 Directory created/verified: {export_dir}")

# Export the cleaned dataset
try:
    swac_fb.to_csv(export_path, index=False)
    print(f"✅ Dataset exported successfully!")
    print(f"   📄 File: {export_filename}")
    print(f"   📍 Location: {export_dir}")
    print(f"   📊 Records exported: {len(swac_fb):,}")
    print(f"   🎯 Success rate: {((len(swac_fb) - swac_fb['player_state'].isnull().sum()) / len(swac_fb) * 100):.2f}%")
    
    # Verify file was created
    if os.path.exists(export_path):
        file_size = os.path.getsize(export_path) / (1024 * 1024)  # Size in MB
        print(f"   📏 File size: {file_size:.1f} MB")
        print(f"   ✅ File verified at: {export_path}")
    
except Exception as e:
    print(f"❌ Error exporting dataset: {str(e)}")
    print(f"   Please check that the directory path is accessible")

print(f"\n🎉 EXPORT COMPLETE!")
print(f"Your cleaned SWAC dataset is ready for analysis!")

💾 EXPORTING CLEANED SWAC DATASET
📁 Directory created/verified: C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2
✅ Dataset exported successfully!
   📄 File: SWAC_rosters_clean.csv
   📍 Location: C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2
   📊 Records exported: 15,571
   🎯 Success rate: 99.46%
   📏 File size: 1.2 MB
   ✅ File verified at: C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2\SWAC_rosters_clean.csv

🎉 EXPORT COMPLETE!
Your cleaned SWAC dataset is ready for analysis!


In [27]:
# ===== VIEW REMAINING MISSING PLAYERS =====

print("📋 REMAINING 93 MISSING PLAYERS - DETAILED VIEW")
print("=" * 60)

# Get all remaining missing players
remaining_missing = swac_fb[swac_fb['player_state'].isnull()].copy()

print(f"Total remaining missing: {len(remaining_missing)} players")
print(f"Current success rate: {((len(swac_fb) - len(remaining_missing)) / len(swac_fb) * 100):.2f}%")

# Create a clean view for analysis
missing_df = remaining_missing[['name', 'team', 'season', 'hometown', 'high_school', 'previous_school']].copy()

# Sort by team and season for easier viewing
missing_df = missing_df.sort_values(['team', 'season', 'name'])

print(f"\n🏫 Breakdown by team:")
team_counts = missing_df['team'].value_counts()
for team, count in team_counts.items():
    total_team = len(swac_fb[swac_fb['team'] == team])
    pct = (count / total_team) * 100
    print(f"   {team:20s}: {count:2d} missing ({pct:.1f}%)")

print(f"\n📅 Breakdown by season:")
season_counts = missing_df['season'].value_counts().sort_index()
print(season_counts.to_string())

# Display the full dataframe
print(f"\n📊 COMPLETE LIST OF REMAINING MISSING PLAYERS:")
print("=" * 120)
missing_df

📋 REMAINING 93 MISSING PLAYERS - DETAILED VIEW
Total remaining missing: 93 players
Current success rate: 99.40%

🏫 Breakdown by team:
   Alabama A&M         : 24 missing (2.0%)
   Bethune-Cookman     : 17 missing (1.2%)
   Southern            : 14 missing (1.1%)
   Alabama State       :  9 missing (0.6%)
   Alcorn State        :  8 missing (0.6%)
   Florida A&M         :  7 missing (0.5%)
   Prairie View A&M    :  5 missing (0.3%)
   Grambling           :  4 missing (0.3%)
   UAPB                :  3 missing (0.3%)
   Texas Southern      :  2 missing (0.1%)

📅 Breakdown by season:
season
2010     2
2011    11
2012     3
2013    13
2014     4
2015    10
2016     4
2017     2
2018     2
2020     7
2021    10
2022     7
2023     6
2024     7
2025     5

📊 COMPLETE LIST OF REMAINING MISSING PLAYERS:


,name,team,season,hometown,high_school,previous_school
3393,Averee Giles,Alabama A&M,2014,NaN,NaN,None
3365,Conard Johnson,Alabama A&M,2014,NaN,NaN,None
3394,Kalian Jackson,Alabama A&M,2014,NaN,NaN,None
3353,Lorenzo Jackson,Alabama A&M,2014,NaN,NaN,None
3104,Denzel Davis,Alabama A&M,2017,NaN,NaN,None
...,...,...,...,...,...,...
7662,Costley Williams,Texas Southern,2010,None,NaN,None
7605,Jarbar Perkins,Texas Southern,2010,None,NaN,None
8541,De'Marte Darrett,UAPB,2016,None,NaN,None
7964,Grant Ewell Jr,UAPB,2022,"Portland, OR.",NaN,Butte CC


In [33]:
# ===== ANALYSIS: HIGH SCHOOL DATA ENHANCEMENT =====

print("🏫 ANALYZING HIGH SCHOOL DATA ENHANCEMENT OPPORTUNITY")
print("=" * 60)

# Current high school data coverage
current_hs_coverage = swac_fb['high_school'].notna().sum()
total_players = len(swac_fb)
current_percentage = (current_hs_coverage / total_players) * 100

print(f"📊 CURRENT HIGH SCHOOL DATA COVERAGE:")
print(f"   Players with high school data: {current_hs_coverage:,} ({current_percentage:.1f}%)")
print(f"   Players missing high school data: {total_players - current_hs_coverage:,}")

# Define patterns for high school identification
hs_patterns = [
    r'\bHS$',                    # Ends with "HS"
    r'\bHigh School$',           # Ends with "High School"
    r'\b\w+\s+High$',           # Something + "High" (e.g., "Central High")
    r'\bAcademy$',              # Ends with "Academy"
    r'\bCharter$',              # Ends with "Charter"
    r'\bPrep$'                  # Ends with "Prep"
]

# Military/College academies to exclude
military_colleges = [
    'Air Force Academy', 'Naval Academy', 'US Military Academy',
    'Coast Guard Academy', 'Merchant Marine Academy'
]

# Problematic team/year combinations to skip
skip_conditions = [
    ('Prairie View A&M', 2010),
    # Add early Texas Southern years if needed
]

print(f"\n🔍 IDENTIFYING CANDIDATES FOR HIGH SCHOOL DATA TRANSFER:")
print(f"Patterns to match: {len(hs_patterns)} high school patterns")
print(f"Military colleges to exclude: {len(military_colleges)}")
print(f"Team/year combinations to skip: {len(skip_conditions)}")

# Analyze potential transfers
candidates = []

for idx, row in swac_fb.iterrows():
    # Skip if high_school is already filled
    if pd.notna(row['high_school']) and row['high_school'].strip():
        continue
    
    # Skip if previous_school is empty
    if pd.isna(row['previous_school']) or not row['previous_school'].strip():
        continue
    
    # Skip problematic team/year combinations
    skip_this = False
    for team, year in skip_conditions:
        if row['team'] == team and row['season'] == year:
            skip_this = True
            break
    if skip_this:
        continue
    
    prev_school = str(row['previous_school']).strip()
    
    # Skip if it's a military college
    if any(mil_college.lower() in prev_school.lower() for mil_college in military_colleges):
        continue
    
    # Skip if previous_school equals current high_school (Prairie View duplicates)
    if pd.notna(row['high_school']) and prev_school == str(row['high_school']).strip():
        continue
    
    # Check if it matches any high school pattern
    matches_pattern = False
    matched_pattern = ""
    
    for pattern in hs_patterns:
        if re.search(pattern, prev_school, re.IGNORECASE):
            matches_pattern = True
            matched_pattern = pattern
            break
    
    if matches_pattern:
        candidates.append({
            'index': idx,
            'name': row['name'],
            'team': row['team'],
            'season': row['season'],
            'current_high_school': row['high_school'],
            'previous_school': prev_school,
            'pattern_matched': matched_pattern
        })

print(f"\n📋 ANALYSIS RESULTS:")
print(f"   Candidates found: {len(candidates)}")

if len(candidates) > 0:
    # Show breakdown by pattern
    pattern_counts = {}
    for candidate in candidates:
        pattern = candidate['pattern_matched']
        pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1
    
    print(f"\n🎯 BREAKDOWN BY PATTERN:")
    for pattern, count in pattern_counts.items():
        print(f"   {pattern}: {count} candidates")
    
    # Show breakdown by team
    team_counts = {}
    for candidate in candidates:
        team = candidate['team']
        team_counts[team] = team_counts.get(team, 0) + 1
    
    print(f"\n🏫 BREAKDOWN BY TEAM:")
    for team, count in sorted(team_counts.items()):
        print(f"   {team}: {count} candidates")
    
    # Show sample of candidates
    print(f"\n📋 SAMPLE OF CANDIDATES (first 20):")
    print("-" * 120)
    print(f"{'Name':<25} {'Team':<20} {'Season':<6} {'Previous School':<40} {'Pattern'}")
    print("-" * 120)
    
    for i, candidate in enumerate(candidates[:20]):
        print(f"{candidate['name'][:24]:<25} {candidate['team']:<20} {candidate['season']:<6} {candidate['previous_school'][:39]:<40} {candidate['pattern_matched']}")
    
    if len(candidates) > 20:
        print(f"... and {len(candidates) - 20} more candidates")
    
    # Calculate potential improvement
    new_hs_coverage = current_hs_coverage + len(candidates)
    new_percentage = (new_hs_coverage / total_players) * 100
    improvement = new_percentage - current_percentage
    
    print(f"\n📈 POTENTIAL IMPROVEMENT:")
    print(f"   Current coverage: {current_percentage:.1f}%")
    print(f"   After enhancement: {new_percentage:.1f}%")
    print(f"   Improvement: +{improvement:.1f} percentage points")
    print(f"   Additional high schools: {len(candidates)}")

else:
    print("   No candidates found with current criteria")

print(f"\n💡 Ready to proceed? This analysis shows what would be moved.")

🏫 ANALYZING HIGH SCHOOL DATA ENHANCEMENT OPPORTUNITY
📊 CURRENT HIGH SCHOOL DATA COVERAGE:
   Players with high school data: 7,213 (46.3%)
   Players missing high school data: 8,358

🔍 IDENTIFYING CANDIDATES FOR HIGH SCHOOL DATA TRANSFER:
Patterns to match: 6 high school patterns
Military colleges to exclude: 5
Team/year combinations to skip: 1

📋 ANALYSIS RESULTS:
   Candidates found: 3326

🎯 BREAKDOWN BY PATTERN:
   \bHS$: 3027 candidates
   \bAcademy$: 78 candidates
   \bPrep$: 31 candidates
   \bHigh School$: 30 candidates
   \b\w+\s+High$: 156 candidates
   \bCharter$: 4 candidates

🏫 BREAKDOWN BY TEAM:
   Alabama A&M: 345 candidates
   Alabama State: 847 candidates
   Alcorn State: 127 candidates
   Florida A&M: 288 candidates
   Grambling: 837 candidates
   Jackson State: 127 candidates
   Prairie View A&M: 143 candidates
   Southern: 1 candidates
   Texas Southern: 7 candidates
   UAPB: 604 candidates

📋 SAMPLE OF CANDIDATES (first 20):
------------------------------------------

In [34]:
# ===== IMPLEMENTING HIGH SCHOOL DATA ENHANCEMENT =====

print("🚀 IMPLEMENTING HIGH SCHOOL DATA ENHANCEMENT")
print("=" * 50)

# Store starting statistics
starting_hs_coverage = swac_fb['high_school'].notna().sum()
starting_percentage = (starting_hs_coverage / len(swac_fb)) * 100

print(f"📊 STARTING STATISTICS:")
print(f"   High school coverage: {starting_hs_coverage:,} ({starting_percentage:.1f}%)")

# Define patterns and exclusions (same as analysis)
hs_patterns = [
    r'\bHS$',                    # Ends with "HS"
    r'\bHigh School$',           # Ends with "High School"
    r'\b\w+\s+High$',           # Something + "High" (e.g., "Central High")
    r'\bAcademy$',              # Ends with "Academy"
    r'\bCharter$',              # Ends with "Charter"
    r'\bPrep$'                  # Ends with "Prep"
]

military_colleges = [
    'Air Force Academy', 'Naval Academy', 'US Military Academy',
    'Coast Guard Academy', 'Merchant Marine Academy'
]

skip_conditions = [
    ('Prairie View A&M', 2010),
]

# Perform the transfer
transfers_made = 0
transfer_log = []

print(f"\n🔧 PERFORMING HIGH SCHOOL DATA TRANSFERS:")

for idx, row in swac_fb.iterrows():
    # Skip if high_school is already filled
    if pd.notna(row['high_school']) and str(row['high_school']).strip():
        continue
    
    # Skip if previous_school is empty
    if pd.isna(row['previous_school']) or not str(row['previous_school']).strip():
        continue
    
    # Skip problematic team/year combinations
    skip_this = False
    for team, year in skip_conditions:
        if row['team'] == team and row['season'] == year:
            skip_this = True
            break
    if skip_this:
        continue
    
    prev_school = str(row['previous_school']).strip()
    
    # Skip if it's a military college
    if any(mil_college.lower() in prev_school.lower() for mil_college in military_colleges):
        continue
    
    # Skip if previous_school equals current high_school (Prairie View duplicates)
    if pd.notna(row['high_school']) and prev_school == str(row['high_school']).strip():
        continue
    
    # Check if it matches any high school pattern
    matches_pattern = False
    matched_pattern = ""
    
    for pattern in hs_patterns:
        if re.search(pattern, prev_school, re.IGNORECASE):
            matches_pattern = True
            matched_pattern = pattern
            break
    
    if matches_pattern:
        # Perform the transfer
        swac_fb.loc[idx, 'high_school'] = prev_school
        # Clear the previous_school since it was actually their high school
        swac_fb.loc[idx, 'previous_school'] = None
        
        transfers_made += 1
        transfer_log.append({
            'index': idx,
            'name': row['name'],
            'team': row['team'],
            'season': row['season'],
            'transferred_school': prev_school,
            'pattern': matched_pattern
        })

# Calculate final statistics
final_hs_coverage = swac_fb['high_school'].notna().sum()
final_percentage = (final_hs_coverage / len(swac_fb)) * 100
improvement = final_percentage - starting_percentage

print(f"   ✅ Transfers completed: {transfers_made:,}")

print(f"\n📈 FINAL RESULTS:")
print(f"   Starting coverage: {starting_percentage:.1f}%")
print(f"   Final coverage: {final_percentage:.1f}%")
print(f"   Improvement: +{improvement:.1f} percentage points")
print(f"   Additional high schools: {transfers_made:,}")

# Show breakdown by pattern
if transfers_made > 0:
    pattern_breakdown = {}
    team_breakdown = {}
    
    for transfer in transfer_log:
        pattern = transfer['pattern']
        team = transfer['team']
        pattern_breakdown[pattern] = pattern_breakdown.get(pattern, 0) + 1
        team_breakdown[team] = team_breakdown.get(team, 0) + 1
    
    print(f"\n🎯 TRANSFERS BY PATTERN:")
    for pattern, count in sorted(pattern_breakdown.items(), key=lambda x: x[1], reverse=True):
        print(f"   {pattern}: {count:,} transfers")
    
    print(f"\n🏫 TRANSFERS BY TEAM:")
    for team, count in sorted(team_breakdown.items(), key=lambda x: x[1], reverse=True):
        print(f"   {team}: {count:,} transfers")
    
    # Show sample of transfers made
    print(f"\n📋 SAMPLE OF TRANSFERS MADE (first 10):")
    print("-" * 100)
    print(f"{'Name':<25} {'Team':<20} {'Season':<6} {'High School Transferred'}")
    print("-" * 100)
    
    for i, transfer in enumerate(transfer_log[:10]):
        print(f"{transfer['name'][:24]:<25} {transfer['team']:<20} {transfer['season']:<6} {transfer['transferred_school']}")
    
    if len(transfer_log) > 10:
        print(f"... and {len(transfer_log) - 10:,} more transfers")

print(f"\n🎉 HIGH SCHOOL DATA ENHANCEMENT COMPLETE!")
print(f"The dataset now has significantly improved high school coverage!")

🚀 IMPLEMENTING HIGH SCHOOL DATA ENHANCEMENT
📊 STARTING STATISTICS:
   High school coverage: 7,213 (46.3%)

🔧 PERFORMING HIGH SCHOOL DATA TRANSFERS:
   ✅ Transfers completed: 3,326

📈 FINAL RESULTS:
   Starting coverage: 46.3%
   Final coverage: 67.7%
   Improvement: +21.4 percentage points
   Additional high schools: 3,326

🎯 TRANSFERS BY PATTERN:
   \bHS$: 3,027 transfers
   \b\w+\s+High$: 156 transfers
   \bAcademy$: 78 transfers
   \bPrep$: 31 transfers
   \bHigh School$: 30 transfers
   \bCharter$: 4 transfers

🏫 TRANSFERS BY TEAM:
   Alabama State: 847 transfers
   Grambling: 837 transfers
   UAPB: 604 transfers
   Alabama A&M: 345 transfers
   Florida A&M: 288 transfers
   Prairie View A&M: 143 transfers
   Jackson State: 127 transfers
   Alcorn State: 127 transfers
   Texas Southern: 7 transfers
   Southern: 1 transfers

📋 SAMPLE OF TRANSFERS MADE (first 10):
----------------------------------------------------------------------------------------------------
Name                

In [36]:
# ===== RE-EXPORT ENHANCED DATASET =====

print("💾 RE-EXPORTING ENHANCED SWAC DATASET")
print("=" * 45)

# Define export path and filename (same as before)
export_dir = r"C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2"
export_filename = "SWAC_rosters_clean.csv"
export_path = os.path.join(export_dir, export_filename)

# Get current statistics
current_success_rate = ((len(swac_fb) - swac_fb['player_state'].isnull().sum()) / len(swac_fb) * 100)
current_hs_coverage = swac_fb['high_school'].notna().sum()
hs_percentage = (current_hs_coverage / len(swac_fb)) * 100

print(f"📊 ENHANCED DATASET STATISTICS:")
print(f"   Total records: {len(swac_fb):,}")
print(f"   Player state success rate: {current_success_rate:.2f}%")
print(f"   High school coverage: {current_hs_coverage:,} ({hs_percentage:.1f}%)")
print(f"   Missing player states: {swac_fb['player_state'].isnull().sum()}")

# Export the enhanced dataset
try:
    swac_fb.to_csv(export_path, index=False)
    print(f"\n✅ ENHANCED DATASET EXPORTED SUCCESSFULLY!")
    print(f"   📄 File: {export_filename}")
    print(f"   📍 Location: {export_dir}")
    print(f"   🔄 Overwrote previous version")
    
    # Verify file was created and get size
    if os.path.exists(export_path):
        file_size = os.path.getsize(export_path) / (1024 * 1024)  # Size in MB
        print(f"   📏 File size: {file_size:.1f} MB")
        print(f"   ✅ File verified at: {export_path}")
        
        # Show what's improved
        print(f"\n🎯 ENHANCEMENTS IN THIS VERSION:")
        print(f"   ✅ Player state assignments: 99.46% complete")
        print(f"   ✅ High school data: 67.7% complete (+21.4% improvement)")
        print(f"   ✅ 3,326 additional high school records")
        print(f"   ✅ International players properly flagged")
        print(f"   ✅ Team state mappings: 100% complete")
    
except Exception as e:
    print(f"❌ Error exporting enhanced dataset: {str(e)}")
    print(f"   Please check that the directory path is accessible")

print(f"\n🎉 ENHANCED EXPORT COMPLETE!")
print(f"Your dataset now has both excellent state coverage AND high school coverage!")

💾 RE-EXPORTING ENHANCED SWAC DATASET
📊 ENHANCED DATASET STATISTICS:
   Total records: 15,571
   Player state success rate: 99.46%
   High school coverage: 10,539 (67.7%)
   Missing player states: 84

✅ ENHANCED DATASET EXPORTED SUCCESSFULLY!
   📄 File: SWAC_rosters_clean.csv
   📍 Location: C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2
   🔄 Overwrote previous version
   📏 File size: 1.2 MB
   ✅ File verified at: C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2\SWAC_rosters_clean.csv

🎯 ENHANCEMENTS IN THIS VERSION:
   ✅ Player state assignments: 99.46% complete
   ✅ High school data: 67.7% complete (+21.4% improvement)
   ✅ 3,326 additional high school records
   ✅ International players properly flagged
   ✅ Team state mappings: 100% complete

🎉 ENHANCED EXPORT COMPLETE!
Your dataset now has both excellent state coverage AND high school coverage!


In [37]:
# ===== IMPORT MANUALLY-CORRECTED CSV AND FINAL ANALYSIS =====

print("📥 IMPORTING MANUALLY-CORRECTED DATASET")
print("=" * 50)

# Import the manually-corrected CSV
csv_path = r"C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters2\SWAC_rosters_clean.csv"

try:
    swac_final = pd.read_csv(csv_path)
    print(f"✅ Successfully imported manually-corrected dataset!")
    print(f"   📊 Total records: {len(swac_final):,}")
    print(f"   📋 Columns: {list(swac_final.columns)}")
    
    # Calculate final statistics
    print(f"\n📈 FINAL DATA QUALITY ANALYSIS:")
    print("=" * 40)
    
    # Player state coverage
    state_coverage = swac_final['player_state'].notna().sum()
    state_percentage = (state_coverage / len(swac_final)) * 100
    missing_states = swac_final['player_state'].isnull().sum()
    
    print(f"🗺️ PLAYER STATE ASSIGNMENTS:")
    print(f"   ✅ Complete: {state_coverage:,} ({state_percentage:.2f}%)")
    print(f"   ❌ Missing: {missing_states}")
    print(f"   🌍 International: {(swac_final['player_state'] == 'INTL').sum()}")
    
    # High school coverage
    hs_coverage = swac_final['high_school'].notna().sum()
    hs_percentage = (hs_coverage / len(swac_final)) * 100
    missing_hs = len(swac_final) - hs_coverage
    
    print(f"\n🏫 HIGH SCHOOL DATA COVERAGE:")
    print(f"   ✅ Complete: {hs_coverage:,} ({hs_percentage:.1f}%)")
    print(f"   ❌ Missing: {missing_hs:,}")
    
    # Compare to our automated pipeline results
    print(f"\n📊 IMPROVEMENT FROM MANUAL CORRECTIONS:")
    automated_hs_coverage = 67.7  # From our pipeline
    manual_improvement = hs_percentage - automated_hs_coverage
    print(f"   🤖 Automated pipeline: {automated_hs_coverage:.1f}%")
    print(f"   ✋ After manual corrections: {hs_percentage:.1f}%")
    print(f"   📈 Manual improvement: +{manual_improvement:.1f} percentage points")
    
    # Check if we hit the 95% target
    target_hs = 95.0
    if hs_percentage >= target_hs:
        print(f"\n🎉 TARGET ACHIEVED! High school coverage ({hs_percentage:.1f}%) exceeds {target_hs}% goal!")
    else:
        remaining_needed = int((target_hs/100 * len(swac_final)) - hs_coverage)
        print(f"\n🎯 Progress toward 95% target:")
        print(f"   Current: {hs_percentage:.1f}%")
        print(f"   Target: {target_hs}%")
        print(f"   Still need: {remaining_needed} more high schools")
    
    # Team-level analysis
    print(f"\n🏫 HIGH SCHOOL COVERAGE BY TEAM:")
    team_hs_stats = []
    for team in sorted(swac_final['team'].unique()):
        team_data = swac_final[swac_final['team'] == team]
        team_hs_complete = team_data['high_school'].notna().sum()
        team_total = len(team_data)
        team_hs_rate = (team_hs_complete / team_total * 100) if team_total > 0 else 0
        team_hs_stats.append((team, team_hs_rate, team_hs_complete, team_total))
        print(f"   {team:20s}: {team_hs_rate:5.1f}% ({team_hs_complete:,} of {team_total:,})")
    
    # Show sample of final data
    print(f"\n📋 SAMPLE OF FINAL CLEANED DATA:")
    sample_cols = ['name', 'team', 'season', 'hometown', 'player_state', 'high_school']
    if all(col in swac_final.columns for col in sample_cols):
        print(swac_final[sample_cols].head(10).to_string(index=False))
    
    print(f"\n🎉 FINAL DATASET ANALYSIS COMPLETE!")
    print(f"Dataset is ready for recruitment pattern analysis!")
    
except Exception as e:
    print(f"❌ Error importing CSV: {str(e)}")
    print(f"   Please check the file path and ensure the CSV was saved correctly")

📥 IMPORTING MANUALLY-CORRECTED DATASET
✅ Successfully imported manually-corrected dataset!
   📊 Total records: 15,571
   📋 Columns: ['team', 'season', 'name', 'high_school', 'hometown', 'previous_school', 'class', 'team_state', 'player_state', 'is_international']

📈 FINAL DATA QUALITY ANALYSIS:
🗺️ PLAYER STATE ASSIGNMENTS:
   ✅ Complete: 15,487 (99.46%)
   ❌ Missing: 84
   🌍 International: 109

🏫 HIGH SCHOOL DATA COVERAGE:
   ✅ Complete: 11,585 (74.4%)
   ❌ Missing: 3,986

📊 IMPROVEMENT FROM MANUAL CORRECTIONS:
   🤖 Automated pipeline: 67.7%
   ✋ After manual corrections: 74.4%
   📈 Manual improvement: +6.7 percentage points

🎯 Progress toward 95% target:
   Current: 74.4%
   Target: 95.0%
   Still need: 3207 more high schools

🏫 HIGH SCHOOL COVERAGE BY TEAM:
   Alabama A&M         :  61.1% (735 of 1,203)
   Alabama State       :  76.1% (1,209 of 1,588)
   Alcorn State        :  82.6% (1,070 of 1,295)
   Bethune-Cookman     :  96.8% (1,408 of 1,455)
   Florida A&M         :  59.5% (898